In [1]:
# ============================================================================
# CELULA 01
# ============================================================================
# SETUP & IMPORTS 
# ============================================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuração de exibição do Pandas
pd.options.display.float_format = '{:,.2f}'.format

def formatar_moeda(valor):
    if isinstance(valor, (int, float)):
        return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return valor

def formatar_pct(valor):
    return f"{valor:.1f}%"

print("✅ Ambiente configurado")
print("📦 Bibliotecas: pandas, numpy, plotly, scipy")
print("🎨 Cores: Movidas para PREMISSAS (Célula 02)")

✅ Ambiente configurado
📦 Bibliotecas: pandas, numpy, plotly, scipy
🎨 Cores: Movidas para PREMISSAS (Célula 02)


In [42]:
# ============================================================================
# CELULA 02: PREMISSAS DO NEGÓCIO (VERSÃO ULTIMATE - FULL CORP)
# ============================================================================

PREMISSAS = {
    # ========================================================================
    # 1. CONFIGURAÇÃO GERAL
    # ========================================================================
    'data_inicio': '2025-11-01',
    'meses_projecao': 36,
    'seed_fixa': 42,
    'semanas_por_mes': 4.33,
    'meses_por_ano': 12,
    
    # ========================================================================
    # 2. RAMP-UP (CURVA DE APRENDIZADO)
    # ========================================================================
    'ramp_up_inicial': 0.60,        
    'ramp_up_incremento': 0.08,     
    
    # ========================================================================
    # 3. FUNIL DE AQUISIÇÃO (B2C)
    # ========================================================================
    'trafego_inicial': 1500,
    'crescimento_trafego_mes_1_6': 0.20,
    'crescimento_trafego_mes_7_12': 0.08,
    'crescimento_trafego_mes_13_plus': 0.05,
    'taxa_visitante_para_trial': 0.04,
    'taxa_trial_para_pagante': 0.12,
    'usuarios_pagos_iniciais': 20,
    
    # ========================================================================
    # 4. ORGÂNICO & BRANDING
    # ========================================================================
    'fator_visitas_organicas_por_pagante': 100,
    'taxa_crescimento_organico_base': 0.02,
    'elasticidade_organico': 0.70,
    'alerta_dependencia_organica_pct': 50.0,
    'meta_crescimento_mes_1_6': 0.15,
    'meta_crescimento_mes_7_12': 0.08,
    'meta_crescimento_mes_13_plus': 0.05,

    # ========================================================================
    # 5. RETENÇÃO (CHURN)
    # ========================================================================
    'churn_base': 0.08,
    
    # ========================================================================
    # 6. PRICING & MIX
    # ========================================================================
    'preco_lite': 69.90, 'preco_trader': 99.90, 'preco_pro': 169.90,
    'mix_lite': 0.50, 'mix_trader': 0.35, 'mix_pro': 0.15,
    
    # ========================================================================
    # 7. UNIT ECONOMICS (CUSTOS VARIÁVEIS)
    # ========================================================================
    'custo_ia_lite': 1.50, 'custo_ia_trader': 3.60, 'custo_ia_pro': 11.00,
    'custo_ia_por_usuario': 5.50, # Fallback
    'taxa_pagamento': 0.035,
    'taxa_inadimplencia': 0.02,
    'impostos': 0.06,
    
    # ========================================================================
    # 8. INFRAESTRUTURA & TECNOLOGIA (DECOMPOSIÇÃO GRANULAR)
    # ========================================================================
    # NOTA: Valores mensais em R$ (convertidos de USD aprox. 6.0 se necessário)
    
    # --- TIER 1: VALIDAÇÃO (0 a 300 Usuários) ---
    # Estratégia: Monolito Simples (All-in-one) + Dados via MT5
    't1_vps_app_api': 120.00,       # DigitalOcean Droplet (4GB RAM, 2 vCPU) - Backend
    't1_vps_windows_mt5': 180.00,   # AWS t3.medium ou equivalente p/ rodar MetaTrader 24/7
    't1_database_managed': 80.00,   # Supabase/Neon (Plano Pro Starter) ou RDS db.t4g.micro
    't1_storage_s3': 20.00,         # AWS S3 (Backups de logs e dados brutos)
    't1_ferramentas_dev': 300.00,   # GitHub Copilot (x2), Cursor, PyCharm
    't1_observability': 0.00,       # Sentry (Plano Dev Gratuito) / Grafana Self-hosted
    't1_scraping_news': 200.00,     # Firecrawl/Apify (Coleta de notícias)
    't1_dominio_dns': 10.00,        # Cloudflare Free + Registro.br diluído
    't1_email_transacional': 50.00, # AWS SES ou Postmark (Volume baixo)
    # TOTAL TIER 1 PREVISTO: ~R$ 960,00
    
    # --- TIER 2: TRAÇÃO (301 a 1.500 Usuários) ---
    # Estratégia: Separação de Serviços + API Profissional (Fim da Gambiarra MT5)
    't2_api_dados_b3': 5000.00,     # Custo Chave: Cedro/Nelogica API (Non-Display)
    't2_compute_app': 450.00,       # Upgrade: 2x Droplets (Load Balance simples) ou Instance maior
    't2_database_primary': 350.00,  # Upgrade: AWS RDS Postgres (Produção)
    't2_cache_redis': 120.00,       # DigitalOcean Managed Redis (Cache de cotações)
    't2_storage_s3': 100.00,        # Maior volume de logs e dados históricos
    't2_security_waf': 120.00,      # Cloudflare Pro (Proteção básica DDoS)
    't2_ferramentas_dev': 600.00,   # Mais licenças (Equipe crescendo)
    't2_observability': 250.00,     # Sentry Team Plan + Datadog Basic
    't2_suporte_ticket': 300.00,    # Zendesk/Intercom Starter
    # TOTAL TIER 2 PREVISTO: ~R$ 7.290,00
    
    # --- TIER 3: ESCALA (1.501 a 5.000 Usuários) ---
    # Estratégia: Alta Disponibilidade (HA) + Redundância
    't3_api_dados_b3_pro': 7000.00, # Renegociação API (Mais TPS - Throughtput)
    't3_load_balancer': 200.00,     # AWS ELB / DO Load Balancer dedicado
    't3_compute_cluster': 1500.00,  # Cluster de API (3 a 4 instâncias)
    't3_db_primary_replica': 1200.00, # RDS Multi-AZ (Réplica de Leitura obrigatória)
    't3_cache_cluster': 400.00,     # Redis Cluster (Evitar gargalo de sessão)
    't3_data_warehouse': 500.00,    # BigQuery/Snowflake (Início de Analytics pesado)
    't3_security_advanced': 500.00, # WAF Avançado + Pentest recorrente
    't3_ci_cd_pipeline': 300.00,    # GitHub Actions/CircleCI (Minutos de build pagos)
    't3_observability_pro': 1000.00, # Datadog/NewRelic (Monitoramento full stack)
    # TOTAL TIER 3 PREVISTO: ~R$ 12.600,00

    # --- TIER 4: ENTERPRISE (5.000+ Usuários) ---
    # Estratégia: Kubernetes, Big Data & Compliance Bancário
    't4_api_dados_institutional': 10000.00, # Feed Institucional direto ou agregador premium
    't4_k8s_cluster': 4500.00,      # EKS/AKS Cluster (Kubernetes Gerenciado)
    't4_db_aurora_serverless': 3000.00, # AWS Aurora (Escala automática)
    't4_data_lake_engineering': 2000.00, # Processamento Batch (Airflow/Databricks)
    't4_security_soc': 2500.00,     # SIEM, Logs de Auditoria (Compliance)
    't4_support_enterprise': 1500.00, # Planos Enterprise (SLA Garantido) nas ferramentas
    't4_multi_region_backup': 1000.00, # Disaster Recovery (DR)
    # TOTAL TIER 4 PREVISTO: ~R$ 24.500,00
    
    # ========================================================================
    # 9. MARKETING
    # ========================================================================
    'marketing_fixo_mensal': 3000.00,
    'marketing_perc_receita': 0.35,
    'marketing_teto': 25000.00,
    
    'canal_instagram_pct': 0.20, 'canal_facebook_pct': 0.20,
    'canal_youtube_pct': 0.30, 'canal_google_pct': 0.30,
    'cpc_instagram': 0.60, 'cpc_facebook': 0.80, 'cpc_youtube': 2.50, 'cpc_google': 4.50,
    'conv_click_trial_instagram': 0.03, 'conv_click_trial_facebook': 0.04,
    'conv_click_trial_youtube': 0.10, 'conv_click_trial_google': 0.12,
    
    # ========================================================================
    # 10. AFILIADOS
    # ========================================================================
    'modelo_afiliado_habilitado': True,
    'pct_usuarios_via_afiliado': 0.30,
    'comissao_afiliado_tipo': 'primeira_mensalidade',
    'comissao_afiliado_fixo': 50.00, 'comissao_afiliado_pct': 0.20, 'comissao_afiliado_meses': 12,
    
    # ========================================================================
    # 11. EQUIPE & RH
    # ========================================================================
    'salario_fundador': 5000.00, 'threshold_pagar_fundador': 10000.00,
    'salario_dev_senior': 10000.00, 'threshold_contratar_dev_usuarios': 750,
    'salario_cs': 4500.00, 'threshold_contratar_cs_usuarios': 1000,
    'encargos': 0.70,
    
    # ========================================================================
    # 12. CAPITAL
    # ========================================================================
    'capex': 8000.00,
    'capex_recorrente_mes_24': 12000.00, 'mes_capex_recorrente': 24,
    'caixa_inicial': 100.00, 'aporte_mensal': 2000.00, 'meses_aporte': 6,
    'percentual_distribuicao': 0.50, 'split_fundador': 0.50, 'split_investidor': 0.50,
    'alerta_custo_total_receita_pct': 70.0,
    
    # ========================================================================
    # 13. ESCRITÓRIO & ADM BÁSICO
    # ========================================================================
    'threshold_ativar_escritorio': 65000.00,
    'custo_aluguel_condominio': 3500.00, 'custo_internet_redundante': 400.00,
    'custo_utilidades_limpeza': 800.00, 'salario_secretaria_adm': 2500.00,
    'custo_contabilidade': 600.00, 'custo_software_gestao': 400.00,
    'taxa_contingencia_pct': 0.03,

    # ========================================================================
    # 16. GESTÃO CORPORATIVA, GOVERNANÇA & B2B (NOVO - PEDIDO EXPLÍCITO)
    # ========================================================================
    # O nível "Empresa Real" além de SaaS de quarto.
    
    # VIAGENS & REPRESENTAÇÃO
    # Almoços com parceiros, ida a eventos (XP Expert, etc), visitas a corretoras.
    'verba_viagens_representacao_base': 1.00, # Valor fixo mensal se houver receita.
    'verba_viagens_pct_receita': 0.02, # +2% da receita bruta vai para relacionamento comercial.

    # JURÍDICO & COMPLIANCE
    # Revisão de Termos de Uso, LGPD, Contratos com Parceiros.
    'custo_juridico_anual': 1200.00, # R$ 12k/ano gastos pontualmente ou diluídos.
    
    # CONSELHO CONSULTIVO (Advisory Board)
    # Pagar mentores/conselheiros experientes para reuniões trimestrais.
    # Ativa apenas quando a empresa tiver tração real (> R$ 100k MRR).
    'threshold_ativar_conselho': 100000.00, # Gatilho de MRR.
    'jeton_conselheiro_mensal': 3000.00, # Valor pago aos conselheiros (reunião mensal).
    
    # FREELANCERS SAZONAIS (Gig Economy)
    # Design para Black Friday, Copywriter para lançamentos, Video Maker.
    # Acontece a cada 3 meses (Trimestral).
    'custo_freelancer_trimestral': 3000.00, 
    
    # BENEFÍCIOS EXECUTIVOS (Retenção de Talentos/Fundador)
    # Carro da empresa, Plano de Saúde Top, Seguro de Vida.
    # Ativa apenas quando Lucro Líquido > R$ 100k.
    'threshold_beneficios_executivos': 80000.00,
    'custo_carro_saude_mensal': 4500.00, # Leasing Carro + Bradesco Saúde Top.

    # RECEITA B2B (PARCERIAS ESTRATÉGICAS)
    # Venda de "White Label" ou API para uma corretora/mesa proprietária.
    # Eventos raros (ex: 1x por ano) mas de alto valor.
    'probabilidade_fechar_b2b_anual': 0.30, # 30% de chance de fechar um deal B2B/ano.
    'receita_setup_b2b': 15000.00, # Setup Fee cobrado do parceiro.
    'custo_implantacao_b2b': 5000.00, # Custo extra para integrar o parceiro.

    # ========================================================================
    # 14. ANÁLISE DE RISCO
    # ========================================================================
    'mc_n_simulacoes': 1000,
    'mc_prob_atraso_aporte': 0.20,
    'mc_std_churn': 0.30, 'mc_std_vis_trial': 0.30, 'mc_std_trial_pag': 0.30,
    'mc_std_cresc_traf': 0.35, 'mc_std_marketing': 0.15, 'mc_std_infra': 0.15,
    'mc_std_taxa_pag': 0.005, 'mc_std_mix_lite': 0.10,
    'mc_var_ia_otimista': 0.80, 'mc_var_ia_pessimista': 1.50,

    # ========================================================================
    # 15. VISUALIZAÇÃO
    # ========================================================================
    'layout_altura_grafico': 500, 'layout_altura_grafico_large': 600,
    'layout_mrr_zoom_range': 100000, 'layout_mrr_full_range': 800000,
    'janela_cac_meses': 12,
    'benchmark_cdi_anual': 13.0, 'benchmark_ibov_anual': 15.0,
    'benchmark_cac_baixo': 100.0, 'benchmark_cac_alto': 500.0,
    'benchmark_ltv_cac_atencao': 3.0,

    'cores': {
        'revenue': '#36B37E', 'costs': '#FF5630', 'profit': '#0052CC',
        'warning': '#FFAB00', 'neutral': '#6B778C', 'success_light': '#E3FCEF',
        'danger_light': '#FFEBE6', 'primary': '#0052CC', 'success': '#36B37E',
        'danger': '#FF5630', 'marketing_pago': '#3B82F6', 'marketing_organico': '#10B981',
        'dependencia_organica': '#EF4444', 'cogs': '#EF4444', 'marketing_custo': '#F59E0B',
        'infra': '#64748B', 'pessoal': '#3B82F6', 'linha_referencia': '#1E293B',
        'capex_evento': '#DC2626', 'grid': '#E2E8F0', 'texto': '#1E293B'
    }
}
print("✅ PREMISSAS ATUALIZADAS: MODO 'FULL CORP' ATIVADO.")
print("   -> Incluído: Conselho, Jurídico, Viagens, Carro Corp, Freelancers e B2B.")

✅ PREMISSAS ATUALIZADAS: MODO 'FULL CORP' ATIVADO.
   -> Incluído: Conselho, Jurídico, Viagens, Carro Corp, Freelancers e B2B.


In [3]:
# ============================================================================
# CELULA 03: MOTOR DE SIMULAÇÃO (CORRIGIDO E COMPLETO)
# ============================================================================
def simular_projecao(premissas, variacao_params=None, seed=None):
    """
    Motor de projeção financeira completo.
    Garante a exportação de TODAS as colunas de custos para análise detalhada.
    """
    if seed is not None: np.random.seed(seed)
    
    p = premissas.copy()
    if variacao_params:
        for key, valor in variacao_params.items(): p[key] = valor
    
    meses = p['meses_projecao']
    dates = pd.date_range(p['data_inicio'], periods=meses, freq='MS')
    
    # --- 1. INICIALIZAÇÃO DE ARRAYS (Todos os dados mensais) ---
    # Funil & Usuários
    trafego = np.zeros(meses)
    trials = np.zeros(meses, dtype=int)
    novos_pagantes = np.zeros(meses, dtype=int)
    
    usuarios_iniciais = np.zeros(meses)
    usuarios_novos = np.zeros(meses)
    usuarios_perdidos = np.zeros(meses)
    usuarios_finais = np.zeros(meses)
    
    usuarios_lite = np.zeros(meses)
    usuarios_trader = np.zeros(meses)
    usuarios_pro = np.zeros(meses)
    
    # Receita
    receita = np.zeros(meses)
    mrr = np.zeros(meses)
    arpu = np.zeros(meses)
    
    # Custos Variáveis (COGS) - ONDE OCORREU O ERRO ANTERIOR
    custo_ia = np.zeros(meses)
    impostos_arr = np.zeros(meses)
    taxas_pagamento = np.zeros(meses)
    cogs_total = np.zeros(meses)
    
    # Custos Fixos & Operacionais (OPEX)
    infra = np.zeros(meses)
    marketing = np.zeros(meses)
    pessoal = np.zeros(meses)
    comissoes_afiliados = np.zeros(meses)
    
    # Novos Custos (Escritório e Contingência)
    custo_escritorio = np.zeros(meses)
    custo_adm_terceiros = np.zeros(meses)
    custo_contingencia = np.zeros(meses)
    
    opex_total = np.zeros(meses)
    
    # Resultados
    lucro_bruto = np.zeros(meses)
    margem_bruta = np.zeros(meses)
    ebitda = np.zeros(meses) # Aqui igual ao Lucro Operacional
    lucro_liquido = np.zeros(meses)
    
    # Caixa & Investimento
    aportes = np.zeros(meses)
    capex_recorrente = np.zeros(meses)
    distribuicao_fundador = np.zeros(meses)
    distribuicao_investidor = np.zeros(meses)
    caixa = np.zeros(meses)
    
    # Métricas de Marketing (Analytics)
    novos_via_pago_arr = np.zeros(meses, dtype=int)
    novos_via_organico_arr = np.zeros(meses, dtype=int)
    pct_organico_arr = np.zeros(meses)
    deficit_meta_arr = np.zeros(meses, dtype=int)
    
    # Unit Economics
    cac = np.zeros(meses)
    ltv = np.zeros(meses)
    ltv_cac_ratio = np.zeros(meses)
    payback_meses = np.zeros(meses)

    # --- 2. VARIÁVEIS DE ESTADO (Acumuladores e Flags) ---
    usuarios_atual = p['usuarios_pagos_iniciais']
    caixa_atual = p['caixa_inicial'] - p['capex']
    trafego[0] = p['trafego_inicial']
    clientes_afiliados_por_mes = np.zeros(meses, dtype=int)

    # Flags "Sticky" (Para custos não oscilarem)
    flag_prolabore_ativo = False
    flag_dev_ativo = False
    flag_cs_ativo = False
    flag_infra_fase2 = False
    flag_escritorio_ativo = False

    # --- 3. LOOP MENSAL ---
    for t in range(meses):
        mes_num = t + 1
        
        # A. Fluxo Financeiro (Entradas/Saídas de Capital)
        if mes_num <= p['meses_aporte']:
            aportes[t] = p['aporte_mensal']
            caixa_atual += aportes[t]
            
        if mes_num == p['mes_capex_recorrente']:
            capex_recorrente[t] = p['capex_recorrente_mes_24']
            caixa_atual -= capex_recorrente[t]
            
        # B. Marketing Budget
        if mes_num == 1:
            mkt_budget = p['marketing_fixo_mensal']
        else:
            mkt_calc = mrr[t-1] * p['marketing_perc_receita']
            mkt_budget = max(p['marketing_fixo_mensal'], min(mkt_calc, p['marketing_teto']))
            
        # C. Funil de Aquisição (Pago + Orgânico)
        # Cálculo detalhado por canal
        canais = [
            (p['canal_instagram_pct'], p['cpc_instagram'], p['conv_click_trial_instagram']),
            (p['canal_facebook_pct'], p['cpc_facebook'], p['conv_click_trial_facebook']),
            (p['canal_youtube_pct'], p['cpc_youtube'], p['conv_click_trial_youtube']),
            (p['canal_google_pct'], p['cpc_google'], p['conv_click_trial_google'])
        ]
        
        trials_pago_total = 0
        cliques_pago_total = 0
        
        for pct_budget, cpc, conv_rate in canais:
            if cpc > 0:
                budget_canal = mkt_budget * pct_budget
                cliques = budget_canal / cpc
                trials_pago_total += cliques * conv_rate
                cliques_pago_total += cliques
        
        novos_via_pago = int(trials_pago_total * p['taxa_trial_para_pagante'])
        
        # Meta de Crescimento vs Orgânico
        if mes_num <= 6: taxa_meta = p['meta_crescimento_mes_1_6']
        elif mes_num <= 12: taxa_meta = p['meta_crescimento_mes_7_12']
        else: taxa_meta = p['meta_crescimento_mes_13_plus']
        
        meta_novos = int(usuarios_atual * taxa_meta)
        deficit = meta_novos - novos_via_pago
        
        if deficit > 0:
            novos_via_organico = int(deficit * p['elasticidade_organico'])
        else:
            novos_via_organico = int(usuarios_atual * p['taxa_crescimento_organico_base'])
            
        novos_pagantes_raw = novos_via_pago + novos_via_organico
        
        # Analytics Marketing
        novos_via_pago_arr[t] = novos_via_pago
        novos_via_organico_arr[t] = novos_via_organico
        deficit_meta_arr[t] = max(0, deficit)
        if novos_pagantes_raw > 0:
            pct_organico_arr[t] = (novos_via_organico / novos_pagantes_raw) * 100
            
        trafego[t] = cliques_pago_total + (novos_via_organico * p['fator_visitas_organicas_por_pagante'])
        trials[t] = int(trials_pago_total + (novos_via_organico / p['taxa_trial_para_pagante']))
        
        # Ramp-up (Aceleração suave)
        if mes_num <= 6:
            ramp = p['ramp_up_inicial'] + (mes_num - 1) * p['ramp_up_incremento']
            novos_pagantes[t] = int(novos_pagantes_raw * ramp)
        else:
            novos_pagantes[t] = novos_pagantes_raw
            
        # D. Base de Usuários & Receita
        usuarios_iniciais[t] = usuarios_atual
        usuarios_novos[t] = novos_pagantes[t]
        usuarios_perdidos[t] = usuarios_atual * p['churn_base']
        
        usuarios_atual = max(0, usuarios_atual + usuarios_novos[t] - usuarios_perdidos[t])
        usuarios_finais[t] = usuarios_atual
        
        usuarios_lite[t] = usuarios_finais[t] * p['mix_lite']
        usuarios_trader[t] = usuarios_finais[t] * p['mix_trader']
        usuarios_pro[t] = usuarios_finais[t] * p['mix_pro']
        
        receita[t] = (usuarios_lite[t]*p['preco_lite'] + usuarios_trader[t]*p['preco_trader'] + usuarios_pro[t]*p['preco_pro'])
        mrr[t] = receita[t]
        arpu[t] = receita[t] / usuarios_finais[t] if usuarios_finais[t] > 0 else 0
        
        # E. Custos Variáveis (COGS)
        custo_ia[t] = usuarios_finais[t] * p['custo_ia_por_usuario']
        taxas_pagamento[t] = receita[t] * p['taxa_pagamento']
        impostos_arr[t] = receita[t] * p['impostos']
        
        cogs_total[t] = custo_ia[t] + taxas_pagamento[t] + impostos_arr[t]
        lucro_bruto[t] = receita[t] - cogs_total[t]
        margem_bruta[t] = (lucro_bruto[t] / receita[t] * 100) if receita[t] > 0 else 0
        
        # F. Custos Fixos & Estruturais (OPEX)
        
        # F.1 Infraestrutura (Sticky)
        if usuarios_finais[t] > p['threshold_infra_fase2_usuarios']:
            flag_infra_fase2 = True
        infra[t] = p['infra_fase2'] if flag_infra_fase2 else (p['vps_principal'] + p['vps_mt5'] + p['ferramentas_dev'] + p['apis_scraping'] + p['dominio'])
        
        # F.2 Marketing
        marketing[t] = mkt_budget
        
        # F.3 Escritório e Adm (Sticky & Condicional)
        # Gatilho: Lucro Bruto suficiente para pagar estrutura
        if not flag_escritorio_ativo:
            if lucro_bruto[t] > p.get('threshold_ativar_escritorio', 25000):
                flag_escritorio_ativo = True
        
        if flag_escritorio_ativo:
            custo_escritorio[t] = p.get('custo_aluguel_condominio', 3500) + \
                                  p.get('custo_internet_redundante', 600) + \
                                  p.get('custo_utilidades_limpeza', 800)
            # Salário secretária entra na folha abaixo
            custo_secretaria = p.get('salario_secretaria_adm', 2500) * (1 + p['encargos'])
        else:
            custo_escritorio[t] = 0
            custo_secretaria = 0
            
        custo_adm_terceiros[t] = p.get('custo_contabilidade', 500) + p.get('custo_software_gestao', 300)
        custo_contingencia[t] = receita[t] * p.get('taxa_contingencia_pct', 0.03)
        
        # F.4 Afiliados
        if p.get('modelo_afiliado_habilitado', False):
            novos_afiliado = int(novos_pagantes[t] * p['pct_usuarios_via_afiliado'])
            tipo = p['comissao_afiliado_tipo']
            if tipo == 'primeira_mensalidade':
                arpu_ref = arpu[t] if arpu[t] > 0 else 97.0
                comissoes_afiliados[t] = novos_afiliado * arpu_ref
            elif tipo == 'cpa_fixo':
                comissoes_afiliados[t] = novos_afiliado * p['comissao_afiliado_fixo']
            # Recorrente simplificado para brevidade, mas funcional
            elif tipo == 'recorrente':
                clientes_afiliados_por_mes[t] = novos_afiliado
                # Loop simplificado de recorrência
                janela = int(p.get('comissao_afiliado_meses', 12))
                comissao_total = 0
                for m_idx in range(max(0, t-janela+1), t+1):
                    # Estimativa de ativos da coorte
                    ativos = clientes_afiliados_por_mes[m_idx] * ((1-p['churn_base'])**(t-m_idx))
                    arpu_ref = arpu[t] if arpu[t] > 0 else 97.0
                    comissao_total += ativos * arpu_ref * p['comissao_afiliado_pct']
                comissoes_afiliados[t] = comissao_total
                
        # F.5 Pessoal (Sticky - Só contrata, não demite na simulação)
        # Gatilhos
        if mes_num > 1:
            # Usa uma métrica de "Lucro Operacional Estimado" para decidir contratar
            lucro_op_previo = receita[t-1] - cogs_total[t-1] - infra[t-1] - marketing[t-1]
            if not flag_prolabore_ativo:
                if lucro_op_previo > p['threshold_pagar_fundador']:
                    flag_prolabore_ativo = True
        
        if not flag_dev_ativo:
            if usuarios_finais[t] > p['threshold_contratar_dev_usuarios']: flag_dev_ativo = True
            
        if not flag_cs_ativo:
            if usuarios_finais[t] > p['threshold_contratar_cs_usuarios']: flag_cs_ativo = True
            
        # Soma Salários
        folha = 0
        if flag_prolabore_ativo: folha += p['salario_fundador'] * (1 + p['encargos'])
        if flag_dev_ativo: folha += p['salario_dev_senior'] * (1 + p['encargos'])
        if flag_cs_ativo: folha += p['salario_cs'] * (1 + p['encargos'])
        if flag_escritorio_ativo: folha += custo_secretaria
        
        pessoal[t] = folha
        
        # G. Consolidação de Resultados
        opex_total[t] = infra[t] + marketing[t] + pessoal[t] + comissoes_afiliados[t] + \
                        custo_escritorio[t] + custo_adm_terceiros[t] + custo_contingencia[t]
                        
        ebitda[t] = lucro_bruto[t] - opex_total[t]
        lucro_liquido[t] = ebitda[t]
        
        # H. Unit Economics
        janela_cac = p['janela_cac_meses']
        start_idx = max(0, t - (janela_cac - 1))
        custo_aquisicao = marketing[start_idx:t+1].sum() + comissoes_afiliados[start_idx:t+1].sum()
        novos_periodo = novos_pagantes[start_idx:t+1].sum()
        
        cac[t] = (custo_aquisicao / novos_periodo) if novos_periodo > 0 else 0
        
        if receita[t] > 0:
            margem_liq_pct = (receita[t] - cogs_total[t]) / receita[t]
            margem_liq_unit = arpu[t] * margem_liq_pct
            
            ltv[t] = (margem_liq_unit / p['churn_base']) if p['churn_base'] > 0 else 0
            ltv_cac_ratio[t] = (ltv[t] / cac[t]) if cac[t] > 0 else 0
            payback_meses[t] = (cac[t] / margem_liq_unit) if margem_liq_unit > 0 else 0
        
        # I. Caixa
        lucro_retido = lucro_liquido[t]
        if lucro_liquido[t] > 0:
            distr = lucro_liquido[t] * p['percentual_distribuicao']
            distribuicao_fundador[t] = distr * p['split_fundador']
            distribuicao_investidor[t] = distr * p['split_investidor']
            lucro_retido -= distr
            
        caixa_atual += lucro_retido
        caixa[t] = caixa_atual

    # --- 4. EXPORTAÇÃO DO DATAFRAME (COM TODAS AS COLUNAS) ---
    df = pd.DataFrame({
        'Data': dates, 'Mes': range(1, meses + 1),
        # Funil
        'Trafego': trafego.astype(int), 'Trials': trials, 'Novos_Pagantes': novos_pagantes,
        'Novos_Via_Pago': novos_via_pago_arr, 'Novos_Via_Organico': novos_via_organico_arr, 
        'Pct_Organico': pct_organico_arr, 'Deficit_Meta': deficit_meta_arr,
        # Usuários
        'Usuarios_Finais': usuarios_finais, 'Usuarios_Iniciais': usuarios_iniciais, 'Usuarios_Perdidos': usuarios_perdidos,
        'Usuarios_Lite': usuarios_lite, 'Usuarios_Trader': usuarios_trader, 'Usuarios_Pro': usuarios_pro,
        # Receita
        'MRR': mrr, 'Receita_Mensal': receita, 'ARPU': arpu,
        # COGS (DETALHADO - CORRIGINDO O ERRO KEYERROR)
        'Custo_IA': custo_ia, 
        'Impostos': impostos_arr, 
        'Taxas_Pagamento': taxas_pagamento,
        'COGS_Total': cogs_total, 
        'Lucro_Bruto': lucro_bruto, 'Margem_Bruta_Pct': margem_bruta,
        # OPEX (DETALHADO)
        'Infraestrutura': infra, 
        'Marketing': marketing, 
        'Pessoal': pessoal,
        'Comissoes_Afiliados': comissoes_afiliados,
        'Custo_Escritorio': custo_escritorio,          # NOVO
        'Custo_Adm_Terceiros': custo_adm_terceiros,    # NOVO
        'Custo_Contingencia': custo_contingencia,      # NOVO
        'OPEX_Total': opex_total,
        # Resultado
        'EBITDA': ebitda, 'Lucro_Liquido': lucro_liquido,
        # Unit Economics
        'CAC': cac, 'LTV': ltv, 'LTV_CAC_Ratio': ltv_cac_ratio, 'Payback_Meses': payback_meses,
        # Financeiro
        'Dist_Fundador': distribuicao_fundador, 'Dist_Investidor': distribuicao_investidor,
        'Aportes': aportes, 'Capex_Recorrente': capex_recorrente, 'Saldo_Caixa': caixa,
        # Compatibilidade com códigos antigos que buscam colunas "_Ajustado"
        'OPEX_Total_Ajustado': opex_total,
        'Lucro_Liquido_Ajustado': lucro_liquido,
        'Saldo_Caixa_Ajustado': caixa,
        'Dist_Investidor_Ajustado': distribuicao_investidor
    })
    
    return df

print("✅ Motor de Simulação (Célula 03) Refeito Completamente.")
print("   -> Inclui exportação explícita de 'Custo_IA', 'Taxas_Pagamento', 'Impostos'.")
print("   -> Inclui lógica Sticky (sem oscilação de folha).")
print("   -> Inclui Custos de Escritório e Contingência.")

✅ Motor de Simulação (Célula 03) Refeito Completamente.
   -> Inclui exportação explícita de 'Custo_IA', 'Taxas_Pagamento', 'Impostos'.
   -> Inclui lógica Sticky (sem oscilação de folha).
   -> Inclui Custos de Escritório e Contingência.


In [4]:
# ============================================================================
# CELULA 04
# ============================================================================
# FUNÇÕES HELPER: DESCRIÇÕES INTELIGENTES E CONTEXTUALIZADAS
# ============================================================================

def explicar_metrica(nome, valor, unidade='', descricao='', calculo='', 
                     benchmark_min=None, benchmark_max=None):
    """Explica uma métrica de forma contextualizada."""
    print(f"\n{'='*60}")
    print(f"📊 {nome.upper()}")
    print(f"{'='*60}")
    
    if descricao:
        print(f"\n❓ O QUE É:")
        print(f"   {descricao}")
    
    print(f"\n🔢 SEU VALOR:")
    print(f"   {valor} {unidade}")
    
    if calculo:
        print(f"\n🧮 COMO CALCULAMOS:")
        print(f"   {calculo}")
    
    if benchmark_min is not None or benchmark_max is not None:
        print(f"\n📏 BENCHMARK DA INDÚSTRIA (SaaS):")
        if benchmark_min is not None:
            print(f"   Mínimo saudável: {benchmark_min}")
        if benchmark_max is not None:
            print(f"   Excelente: {benchmark_max}")

def interpretar_ltv_cac(ltv, cac, ratio):
    """Interpreta automaticamente a relação LTV/CAC."""
    print(f"\n{'='*60}")
    print("📊 SAÚDE DO NEGÓCIO: Valor do Cliente vs Custo de Aquisição")
    print(f"{'='*60}")
    
    print("\n💰 VALOR DO CLIENTE (LTV)")
    print("   O que é: Quanto dinheiro 1 cliente traz durante toda sua 'vida' conosco")
    print(f"   Seu valor: {formatar_moeda(ltv)}")
    print("   Como calculamos: (Receita mensal × Margem bruta) ÷ Taxa de cancelamento")
    
    print("\n💸 CUSTO DE AQUISIÇÃO (CAC)")
    print("   O que é: Quanto gastamos em marketing para conquistar 1 cliente")
    print(f"   Seu valor: {formatar_moeda(cac)}")
    print("   Como calculamos: (Gasto em marketing últimos 12m) ÷ (Clientes novos últimos 12m)")
    
    if cac < PREMISSAS['benchmark_cac_baixo']:
        print("   ⚠️  ATENÇÃO: CAC muito baixo pode indicar premissas otimistas demais")
    elif cac > PREMISSAS['benchmark_cac_alto']:
        print("   ⚠️  ATENÇÃO: CAC alto - verificar eficiência de marketing")
    else:
        print("   ✅ CAC dentro do esperado para SaaS B2C")
    
    print("\n📊 RELAÇÃO: QUANTO GANHAMOS vs QUANTO GASTAMOS")
    print(f"   Para cada R$ 1,00 gasto em marketing, retorna: R$ {ratio:.2f} em lucro")
    print(f"   Seu valor: {ratio:.1f}× (LTV ÷ CAC)")
    print("   Benchmark SaaS: 3× a 5× é considerado saudável")
    
    if ratio < PREMISSAS['benchmark_ltv_cac_critico']:
        print("\n   🔴 CRÍTICO: Gastamos mais para adquirir do que o cliente retorna!")
        print("      → Ação: Revisar modelo de negócio urgentemente")
    elif ratio < PREMISSAS['benchmark_ltv_cac_atencao']:
        print("\n   🟡 ATENÇÃO: Abaixo do mínimo saudável (3×)")
        print("      → Ação: Reduzir CAC ou aumentar LTV")
    elif ratio <= PREMISSAS['benchmark_ltv_cac_excelente']:
        print("\n   ✅ SAUDÁVEL: Dentro do esperado para SaaS")
        print("      → Continue monitorando e otimizando")
    elif ratio <= PREMISSAS['benchmark_ltv_cac_suspeito']:
        print("\n   🟢 EXCELENTE: Acima da média da indústria!")
        print("      → Oportunidade de investir mais em crescimento")
    else:
        print(f"\n   ⚠️  SUSPEITO: Valor muito alto ({ratio:.1f}×) - verifique premissas")
        print("      → CAC pode estar baixo demais ou LTV alto demais")

def interpretar_payback(payback_meses):
    """Interpreta tempo de payback."""
    payback_semanas = payback_meses * PREMISSAS['semanas_por_mes']
    
    print(f"\n⏱️ TEMPO PARA 'PAGAR' O CLIENTE")
    print(f"   O que é: Quantas semanas até o cliente gerar lucro suficiente para cobrir o CAC")
    print(f"   Seu valor: {payback_semanas:.1f} semanas (~{payback_meses:.1f} meses)")
    print(f"   Benchmark SaaS: 26-52 semanas (6-12 meses) é considerado bom")
    
    if payback_semanas < 13: # Aprox 3 meses
        print(f"   ⚠️  SUSPEITO: Payback muito rápido - verificar premissas")
    elif payback_semanas <= PREMISSAS['benchmark_payback_excelente_sem']:
        print("   ✅ EXCELENTE: Recuperação muito rápida!")
    elif payback_semanas <= PREMISSAS['benchmark_payback_bom_sem']:
        print("   ✅ BOM: Dentro do benchmark")
    else:
        print("   ⚠️  ATENÇÃO: Payback longo - pode dificultar escala")

def interpretar_breakeven(mes_breakeven, caixa_minimo, mes_caixa_minimo):
    """Interpreta break-even e vale da morte."""
    print(f"\n💵 LUCRO E SOBREVIVÊNCIA")
    
    if mes_breakeven:
        semana_breakeven = mes_breakeven * PREMISSAS['semanas_por_mes']
        print(f"\n   📈 BREAK-EVEN (Quando começa a dar lucro)")
        print(f"      Mês: {mes_breakeven} (~{semana_breakeven:.0f} semanas)")
        
        if mes_breakeven <= 6:
            print("      ✅ EXCELENTE: Lucrativo muito rápido!")
        elif mes_breakeven <= 12:
            print("      ✅ BOM: Dentro do esperado")
        elif mes_breakeven <= 18:
            print("      ⚠️  MODERADO: Precisa de capital")
        else:
            print("      🔴 LENTO: Requer muito capital")
    else:
        print("\n   🔴 Break-Even NÃO ATINGIDO em 36 meses")
    
    print(f"\n   💀 VALE DA MORTE (Menor caixa)")
    print(f"      Valor: {formatar_moeda(caixa_minimo)}")
    print(f"      Quando: Mês {mes_caixa_minimo}")
    
    if caixa_minimo < -10000: # Threshold de risco crítico
        print("      🔴 CRÍTICO: Caixa negativo - precisa de mais capital!")
    elif caixa_minimo < 0:
        print("      ⚠️  ATENÇÃO: Momentaneamente negativo")
    else:
        print("      ✅ Caixa sempre positivo")

print("✅ Funções de descrição inteligente carregadas")

✅ Funções de descrição inteligente carregadas


In [30]:
# ============================================================================
# CELULA 05 - EXECUÇÃO DA SIMULAÇÃO BASE E RESUMO
# ============================================================================
print("🔄 Executando simulação base...")

# Roda a simulação usando as premissas atuais (da Célula 02)
df_base = simular_projecao(PREMISSAS, seed=PREMISSAS['seed_fixa'])

# === COLETAR TODOS OS KPIS ===
idx_m12 = PREMISSAS['meses_por_ano'] - 1
idx_m36 = -1 # Último mês

# Funil
trafego_m1 = df_base['Trafego'].iloc[0]
trafego_m12 = df_base['Trafego'].iloc[idx_m12]
trafego_m36 = df_base['Trafego'].iloc[idx_m36]
trials_total = df_base['Trials'].sum()
pagantes_total = df_base['Novos_Pagantes'].sum()

# Usuários
usuarios_m12 = df_base['Usuarios_Finais'].iloc[idx_m12]
usuarios_m36 = df_base['Usuarios_Finais'].iloc[idx_m36]
churn_medio = PREMISSAS['churn_base'] * 100

# Receita
mrr_m12 = df_base['MRR'].iloc[idx_m12]
mrr_m36 = df_base['MRR'].iloc[idx_m36]
arr_m36 = mrr_m36 * 12

# Unit Economics
cac_m12 = df_base['CAC'].iloc[idx_m12]
cac_m36 = df_base['CAC'].iloc[idx_m36]
ltv_m36 = df_base['LTV'].iloc[idx_m36]
ltv_cac_m36 = df_base['LTV_CAC_Ratio'].iloc[idx_m36]
payback_m36 = df_base['Payback_Meses'].iloc[idx_m36]

# Financeiro
mes_breakeven = df_base[df_base['Lucro_Liquido'] > 0]['Mes'].min() if any(df_base['Lucro_Liquido'] > 0) else None
caixa_m12 = df_base['Saldo_Caixa'].iloc[idx_m12]
caixa_m36 = df_base['Saldo_Caixa'].iloc[idx_m36]
caixa_min = df_base['Saldo_Caixa'].min()
mes_caixa_min = df_base[df_base['Saldo_Caixa'] == caixa_min]['Mes'].iloc[0]

# Investidor & ROI (CORREÇÃO DE SEGURANÇA AQUI)
retorno_investidor_total = df_base['Dist_Investidor'].sum()

# Soma o CAPEX + Soma da coluna de Aportes reais do DataFrame
investimento_total = PREMISSAS['capex'] + df_base['Aportes'].sum()

# Cálculo do ROI
if investimento_total > 0:
    roi_pct = ((retorno_investidor_total / investimento_total) - 1) * 100
else:
    roi_pct = 0

print("\n✅ Simulação concluída")
print(f"💰 Investimento Total Considerado na Simulação: {formatar_moeda(investimento_total)}")

# === TABELA RESUMO FORMATADA ===
print("="*80)
print("📊 RESUMO EXECUTIVO DA PROJEÇÃO FINANCEIRA (36 MESES)")
print("="*80)

# Criar DataFrame para tabela
dados_tabela = {
    'Métrica': [],
    'Mês 12 (Ano 1)': [],
    'Mês 36 (Fim)': [],
    'Observação': []
}

# FUNIL
dados_tabela['Métrica'].append('🌐 Tráfego Mensal')
dados_tabela['Mês 12 (Ano 1)'].append(f"{trafego_m12:,.0f}")
dados_tabela['Mês 36 (Fim)'].append(f"{trafego_m36:,.0f}")
dados_tabela['Observação'].append(f"Crescimento vs M1")

dados_tabela['Métrica'].append('👥 Usuários Pagantes')
dados_tabela['Mês 12 (Ano 1)'].append(f"{usuarios_m12:.0f}")
dados_tabela['Mês 36 (Fim)'].append(f"{usuarios_m36:.0f}")
dados_tabela['Observação'].append(f"Total adquiridos: {pagantes_total:.0f}")

# RECEITA
dados_tabela['Métrica'].append('💰 MRR')
dados_tabela['Mês 12 (Ano 1)'].append(formatar_moeda(mrr_m12))
dados_tabela['Mês 36 (Fim)'].append(formatar_moeda(mrr_m36))
dados_tabela['Observação'].append(f"ARR Final: {formatar_moeda(arr_m36)}")

# UNIT ECONOMICS
dados_tabela['Métrica'].append('📊 CAC (Custo Aquisição)')
dados_tabela['Mês 12 (Ano 1)'].append(formatar_moeda(cac_m12))
dados_tabela['Mês 36 (Fim)'].append(formatar_moeda(cac_m36))
dados_tabela['Observação'].append('Rolling 12 meses')

dados_tabela['Métrica'].append('💎 LTV (Valor Cliente)')
dados_tabela['Mês 12 (Ano 1)'].append('-')
dados_tabela['Mês 36 (Fim)'].append(formatar_moeda(ltv_m36))
dados_tabela['Observação'].append(f"LTV/CAC: {ltv_cac_m36:.1f}x")

dados_tabela['Métrica'].append('⏱️ Payback')
dados_tabela['Mês 12 (Ano 1)'].append('-')
dados_tabela['Mês 36 (Fim)'].append(f"{payback_m36:.1f} meses")
dados_tabela['Observação'].append(f"Retorno do CAC")

# FINANCEIRO
dados_tabela['Métrica'].append('💵 Saldo em Caixa')
dados_tabela['Mês 12 (Ano 1)'].append(formatar_moeda(caixa_m12))
dados_tabela['Mês 36 (Fim)'].append(formatar_moeda(caixa_m36))
dados_tabela['Observação'].append(f"Mínimo: {formatar_moeda(caixa_min)}")

dados_tabela['Métrica'].append('🤝 Retorno Investidor')
dados_tabela['Mês 12 (Ano 1)'].append(formatar_moeda(df_base['Dist_Investidor'].iloc[:12].sum()))
dados_tabela['Mês 36 (Fim)'].append(formatar_moeda(retorno_investidor_total))
dados_tabela['Observação'].append(f"ROI: {roi_pct:.1f}% sobre {formatar_moeda(investimento_total)}")

# Exibir tabela
df_resumo = pd.DataFrame(dados_tabela)
print("\n", df_resumo.to_string(index=False))
print("\n" + "="*80)

🔄 Executando simulação base...

✅ Simulação concluída
💰 Investimento Total Considerado na Simulação: R$ 20.000,00
📊 RESUMO EXECUTIVO DA PROJEÇÃO FINANCEIRA (36 MESES)

                 Métrica Mês 12 (Ano 1)  Mês 36 (Fim)                     Observação
       🌐 Tráfego Mensal          4,595        20,314              Crescimento vs M1
    👥 Usuários Pagantes            155          1269         Total adquiridos: 2584
                  💰 MRR   R$ 13.775,30 R$ 112.821,88     ARR Final: R$ 1.353.862,56
📊 CAC (Custo Aquisição)      R$ 222,80     R$ 182,35               Rolling 12 meses
  💎 LTV (Valor Cliente)              -     R$ 936,93                  LTV/CAC: 5.1x
             ⏱️ Payback              -     2.4 meses                 Retorno do CAC
       💵 Saldo em Caixa    R$ 2.673,25 R$ 116.401,14           Mínimo: R$ -9.446,72
   🤝 Retorno Investidor    R$ 3.767,57  R$ 66.631,52 ROI: 233.2% sobre R$ 20.000,00



In [6]:
# ============================================================================
# CELULA 06 - AJUSTE PÓS-SIMULAÇÃO: ADICIONAR CUSTOS DE AFILIADOS
# ============================================================================
if PREMISSAS.get('modelo_afiliado_habilitado', False):
    print("\n🤝 Calculando comissões de afiliados...\n")
    
    # Criar coluna de comissões
    comissoes_afiliados = []
    
    pct_afiliado = PREMISSAS['pct_usuarios_via_afiliado']
    tipo_comissao = PREMISSAS.get('comissao_afiliado_tipo', 'primeira_mensalidade')
    
    for idx, row in df_base.iterrows():
        novos_via_afiliado = int(row['Novos_Pagantes'] * pct_afiliado)
        comissao = 0
        
        if tipo_comissao == 'primeira_mensalidade':
            # ARPU médio * novos clientes afiliados
            comissao = novos_via_afiliado * row['ARPU']
        
        elif tipo_comissao == 'cpa_fixo':
            # Usa valor definido em premissas
            cpa_valor = PREMISSAS.get('comissao_afiliado_fixo', 50.00)
            comissao = novos_via_afiliado * cpa_valor
        
        elif tipo_comissao == 'recorrente':
            # Simplificado: % do ARPU de % dos usuários finais
            pct_comissao = PREMISSAS.get('comissao_afiliado_pct', 0.20)
            usuarios_afiliados = row['Usuarios_Finais'] * pct_afiliado
            comissao = usuarios_afiliados * row['ARPU'] * pct_comissao
        
        comissoes_afiliados.append(comissao)
    
    # Adicionar ao DataFrame
    df_base['Comissao_Afiliados'] = comissoes_afiliados
    
    # Recalcular OPEX e Lucro
    df_base['OPEX_Total_Ajustado'] = df_base['OPEX_Total'] + df_base['Comissao_Afiliados']
    df_base['EBITDA_Ajustado'] = df_base['Lucro_Bruto'] - df_base['OPEX_Total_Ajustado']
    df_base['Lucro_Liquido_Ajustado'] = df_base['EBITDA_Ajustado']
    
    # Recalcular distribuições
    df_base['Dist_Fundador_Ajustado'] = 0.0
    df_base['Dist_Investidor_Ajustado'] = 0.0
    
    for idx, row in df_base.iterrows():
        if row['Lucro_Liquido_Ajustado'] > 0:
            distribuivel = row['Lucro_Liquido_Ajustado'] * PREMISSAS['percentual_distribuicao']
            df_base.at[idx, 'Dist_Fundador_Ajustado'] = distribuivel * PREMISSAS['split_fundador']
            df_base.at[idx, 'Dist_Investidor_Ajustado'] = distribuivel * PREMISSAS['split_investidor']
    
    # Recalcular Caixa
    caixa_ajustado = PREMISSAS['caixa_inicial'] - PREMISSAS['capex']
    caixa_ajustado_arr = []
    
    for idx, row in df_base.iterrows():
        mes_num = idx + 1
        
        # Aportes
        if mes_num <= PREMISSAS['meses_aporte']:
            caixa_ajustado += PREMISSAS['aporte_mensal']
        
        # CAPEX recorrente (Mês parametrizado)
        if mes_num == PREMISSAS['mes_capex_recorrente']:
            caixa_ajustado -= PREMISSAS['capex_recorrente_mes_24']
        
        # Lucro retido
        lucro_retido = row['Lucro_Liquido_Ajustado']
        if row['Lucro_Liquido_Ajustado'] > 0:
            distribuivel = row['Lucro_Liquido_Ajustado'] * PREMISSAS['percentual_distribuicao']
            lucro_retido = row['Lucro_Liquido_Ajustado'] - distribuivel
        
        caixa_ajustado += lucro_retido
        caixa_ajustado_arr.append(caixa_ajustado)
    
    df_base['Saldo_Caixa_Ajustado'] = caixa_ajustado_arr
    
    # Mostrar impacto
    print(f"📊 IMPACTO DOS AFILIADOS:")
    print(f"  Modelo: {tipo_comissao}")
    print(f"  % via afiliados: {pct_afiliado*100:.0f}%")
    print(f"  Comissão total 36m: {formatar_moeda(df_base['Comissao_Afiliados'].sum())}")
    print(f"\n  SEM afiliados:")
    print(f"    • Lucro Líquido M36: {formatar_moeda(df_base['Lucro_Liquido'].iloc[-1])}")
    print(f"    • Retorno Investidor: {formatar_moeda(df_base['Dist_Investidor'].sum())}")
    print(f"    • Caixa Final: {formatar_moeda(df_base['Saldo_Caixa'].iloc[-1])}")
    
    print(f"\n  COM afiliados:")
    print(f"    • Lucro Líquido M36: {formatar_moeda(df_base['Lucro_Liquido_Ajustado'].iloc[-1])}")
    print(f"    • Retorno Investidor: {formatar_moeda(df_base['Dist_Investidor_Ajustado'].sum())}")
    print(f"    • Caixa Final: {formatar_moeda(df_base['Saldo_Caixa_Ajustado'].iloc[-1])}")
    
    print("\n✅ Métricas ajustadas criadas (use colunas '_Ajustado')")
else:
    print("ℹ️ Modelo de afiliados desabilitado (modelo_afiliado_habilitado = False)")

print("✅ CELULA 07: Hardcoded removidos (taxas afiliado, mês capex)")


🤝 Calculando comissões de afiliados...

📊 IMPACTO DOS AFILIADOS:
  Modelo: primeira_mensalidade
  % via afiliados: 30%
  Comissão total 36m: R$ 67.475,10

  SEM afiliados:
    • Lucro Líquido M36: R$ 22.112,06
    • Retorno Investidor: R$ 66.631,52
    • Caixa Final: R$ 116.401,14

  COM afiliados:
    • Lucro Líquido M36: R$ 18.644,96
    • Retorno Investidor: R$ 50.318,43
    • Caixa Final: R$ 81.552,21

✅ Métricas ajustadas criadas (use colunas '_Ajustado')
✅ CELULA 07: Hardcoded removidos (taxas afiliado, mês capex)


In [7]:
# ============================================================================
# CELULA 07 - UNIT ECONOMICS: LTV vs CAC (VISUALIZAÇÃO MELHORADA)
# ============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Dados
meses = df_base['Mes']
ltv = df_base['LTV']
cac = df_base['CAC']
ratio = df_base['LTV_CAC_Ratio']

# Criar figura com eixo duplo
fig_unit = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Barras de LTV e CAC
fig_unit.add_trace(
    go.Bar(x=meses, y=ltv, name="LTV (Valor do Cliente)", marker_color=PREMISSAS['cores']['marketing_organico'], opacity=0.7),
    secondary_y=False
)
fig_unit.add_trace(
    go.Bar(x=meses, y=cac, name="CAC (Custo Aquisição)", marker_color=PREMISSAS['cores']['cogs'], opacity=0.7),
    secondary_y=False
)

# 2. Linha de Ratio (LTV/CAC)
fig_unit.add_trace(
    go.Scatter(x=meses, y=ratio, name="Múltiplo LTV/CAC", 
               line=dict(color=PREMISSAS['cores']['linha_referencia'], width=3, dash='dot'),
               mode='lines+markers'),
    secondary_y=True
)

# 3. Linhas de Referência (Benchmarks)
fig_unit.add_hline(y=PREMISSAS['benchmark_ltv_cac_atencao'], line_dash="dash", line_color=PREMISSAS['cores']['marketing_organico'], secondary_y=True, 
                   annotation_text=f"Meta Saudável ({PREMISSAS['benchmark_ltv_cac_atencao']}x)", annotation_position="top right")
fig_unit.add_hline(y=PREMISSAS['benchmark_ltv_cac_critico'], line_dash="solid", line_color=PREMISSAS['cores']['cogs'], secondary_y=True, 
                   annotation_text="Prejuízo (<1x)", annotation_position="bottom right")

# 4. Anotações Inteligentes (Pontos Chave)
# Mês 12
idx_m12 = PREMISSAS['meses_por_ano'] - 1
if len(meses) >= PREMISSAS['meses_por_ano']:
    r12 = ratio.iloc[idx_m12]
    fig_unit.add_annotation(
        x=PREMISSAS['meses_por_ano'], y=r12, xref="x", yref="y2",
        text=f"M12: {r12:.1f}x", showarrow=True, arrowhead=1, ax=0, ay=-40,
        bgcolor="white", bordercolor=PREMISSAS['cores']['linha_referencia']
    )

# Mês 36 (Final)
r36 = ratio.iloc[-1]
fig_unit.add_annotation(
    x=meses.iloc[-1], y=r36, xref="x", yref="y2",
    text=f"Final: {r36:.1f}x", showarrow=True, arrowhead=1, ax=0, ay=-40,
    bgcolor="white", bordercolor=PREMISSAS['cores']['linha_referencia'], font=dict(size=12, color="black")
)

# Layout Profissional
fig_unit.update_layout(
    title=dict(text="<b>Evolução da Eficiência: LTV vs CAC</b><br><sup>Quanto ganhamos (LTV) vs Quanto gastamos (CAC) por cliente</sup>", font=dict(size=20)),
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    legend=dict(orientation="h", y=1.1),
    margin=dict(t=100)
)

fig_unit.update_yaxes(title_text="Valor Monetário (R$)", secondary_y=False, showgrid=True, gridcolor=PREMISSAS['cores']['grid'])
fig_unit.update_yaxes(title_text="Múltiplo (x)", secondary_y=True, showgrid=False, range=[0, max(10, ratio.max()*1.1)])

fig_unit.show()

# Interpretação
interpretar_ltv_cac(ltv.iloc[-1], cac.iloc[-1], ratio.iloc[-1])
print("✅ CELULA 08: Hardcoded removidos (cores, benchmarks, índices)")


📊 SAÚDE DO NEGÓCIO: Valor do Cliente vs Custo de Aquisição

💰 VALOR DO CLIENTE (LTV)
   O que é: Quanto dinheiro 1 cliente traz durante toda sua 'vida' conosco
   Seu valor: R$ 936,93
   Como calculamos: (Receita mensal × Margem bruta) ÷ Taxa de cancelamento

💸 CUSTO DE AQUISIÇÃO (CAC)
   O que é: Quanto gastamos em marketing para conquistar 1 cliente
   Seu valor: R$ 182,35
   Como calculamos: (Gasto em marketing últimos 12m) ÷ (Clientes novos últimos 12m)
   ✅ CAC dentro do esperado para SaaS B2C

📊 RELAÇÃO: QUANTO GANHAMOS vs QUANTO GASTAMOS
   Para cada R$ 1,00 gasto em marketing, retorna: R$ 5.14 em lucro
   Seu valor: 5.1× (LTV ÷ CAC)
   Benchmark SaaS: 3× a 5× é considerado saudável

   🟢 EXCELENTE: Acima da média da indústria!
      → Oportunidade de investir mais em crescimento
✅ CELULA 08: Hardcoded removidos (cores, benchmarks, índices)


In [8]:
# ============================================================================
# CELULA 08
# ============================================================================
# GRÁFICO 3: TEMPO PARA RECUPERAR INVESTIMENTO EM MARKETING
# ============================================================================

fig_payback = go.Figure()

# Linha principal
fig_payback.add_trace(go.Scatter(
    x=df_base['Mes'],
    y=df_base['Payback_Meses'],
    mode='lines+markers',
    line=dict(color=PREMISSAS['cores']['primary'], width=3),
    marker=dict(size=6, color=PREMISSAS['cores']['primary'], line=dict(width=1, color='white')),
    name='Tempo de Retorno',
    fill='tozeroy',
    fillcolor='rgba(0, 82, 204, 0.1)' # Mantendo alpha hardcoded pois é visual
))

# Linha de referência: Benchmark SaaS
bench_payback = PREMISSAS['benchmark_payback_max_meses']

fig_payback.add_hline(
    y=bench_payback, line_dash="dash", line_color=PREMISSAS['cores']['warning'], line_width=2,
    annotation_text=f"Benchmark SaaS ({bench_payback:.0f} meses)", annotation_position="right"
)

fig_payback.update_layout(
    title={
        'text': '<b>TEMPO PARA RECUPERAR INVESTIMENTO EM MARKETING</b><br>' +
                '<sup>Quantos meses até o cliente "pagar" o custo de aquisição?</sup>',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis=dict(title='<b>Mês</b>'),
    yaxis=dict(
        title='<b>Meses para Recuperar</b>',
        tickformat='.1f',
        range=[0, min(bench_payback * 2, df_base['Payback_Meses'].max() * 1.1)]
    ),
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    font=dict(size=12),
    margin=dict(t=120, b=80, l=80, r=80)
)

fig_payback.show()

# Análise Inteligente
payback_medio = df_base['Payback_Meses'].mean()
payback_final = df_base['Payback_Meses'].iloc[-1]
meses_abaixo_bench = (df_base['Payback_Meses'] <= bench_payback).sum()
pct_abaixo_bench = (meses_abaixo_bench / len(df_base)) * 100

print(f"\n⏱️ TEMPO PARA RECUPERAR INVESTIMENTO")
print(f"  • Tempo Médio: {payback_medio:.1f} meses")
print(f"  • Tempo Final (M36): {payback_final:.1f} meses")
print(f"  • Meses dentro do benchmark (≤{bench_payback:.0f}m): {meses_abaixo_bench}/{len(df_base)} ({pct_abaixo_bench:.0f}%)")
print(f"  • Status: {'✅ Excelente (≤Meta)' if payback_final <= bench_payback else '⚠️ Atenção (>Meta)'}")
print("✅ CELULA 09: Hardcoded removidos (cores, benchmark payback)")


⏱️ TEMPO PARA RECUPERAR INVESTIMENTO
  • Tempo Médio: 2.9 meses
  • Tempo Final (M36): 2.4 meses
  • Meses dentro do benchmark (≤12m): 36/36 (100%)
  • Status: ✅ Excelente (≤Meta)
✅ CELULA 09: Hardcoded removidos (cores, benchmark payback)


In [9]:
# ============================================================================
# CELULA 9
# ============================================================================
# GRÁFICO 4: MARKETING ANALYTICS (PAGO vs ORGÂNICO)
# ============================================================================

fig_mkt = go.Figure()

# 1. Barras Empilhadas: Pago vs Orgânico
fig_mkt.add_trace(go.Bar(
    x=df_base['Mes'],
    y=df_base['Novos_Via_Pago'],
    name='Via Tráfego Pago',
    marker_color=PREMISSAS['cores']['marketing_pago']
))

fig_mkt.add_trace(go.Bar(
    x=df_base['Mes'],
    y=df_base['Novos_Via_Organico'],
    name='Via Orgânico (Necessário)',
    marker_color=PREMISSAS['cores']['marketing_organico']
))

# 2. Linha de % Orgânico (Eixo Secundário)
# Mostra o quão dependente somos do orgânico
fig_mkt.add_trace(go.Scatter(
    x=df_base['Mes'],
    y=df_base['Pct_Organico'],
    name='% Dependência Orgânica',
    yaxis='y2',
    mode='lines+markers',
    line=dict(color=PREMISSAS['cores']['dependencia_organica'], width=2, dash='dot')
))

fig_mkt.update_layout(
    title={
        'text': '<b>ORIGEM DO CRESCIMENTO: PAGO vs ORGÂNICO</b><br>' +
                '<sup>Quanto do crescimento depende de budget vs esforço orgânico?</sup>',
        'x': 0.5,
        'xanchor': 'center'
    },
    barmode='stack',
    xaxis=dict(title='<b>Mês</b>'),
    yaxis=dict(title='<b>Novos Usuários</b>'),
    yaxis2=dict(
        title='<b>% Orgânico</b>',
        overlaying='y',
        side='right',
        range=[0, 100],
        showgrid=False
    ),
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    legend=dict(orientation='h', y=1.1),
    margin=dict(t=120)
)

fig_mkt.show()

# Análise de Risco
pct_organico_medio = df_base['Pct_Organico'].mean()
deficit_total = df_base['Deficit_Meta'].sum()
threshold_org = PREMISSAS.get('alerta_dependencia_organica_pct', 50.0)

print(f"\n📊 ANÁLISE DE DEPENDÊNCIA ORGÂNICA")
print(f"  • Dependência Média: {pct_organico_medio:.1f}% do crescimento vem do orgânico")
print(f"  • Déficit de Meta (36m): {deficit_total} usuários precisaram vir do orgânico")

if pct_organico_medio > threshold_org:
    print(f"  ⚠️ ALERTA DE RISCO: Alta dependência de tráfego orgânico (>{threshold_org:.0f}%)")
    print("     Seu budget de marketing atual é INSUFICIENTE para atingir a meta sozinho.")
    print("     Ação: Aumentar budget ou garantir estratégia de SEO/Conteúdo impecável.")
else:
    print("  ✅ SAUDÁVEL: Budget de marketing suporta a maior parte do crescimento.")
    print("     O orgânico é um bônus para reduzir CAC, não uma necessidade vital.")

# Detalhe por canal (Mês 12)
idx_m12 = PREMISSAS['meses_por_ano'] - 1
print(f"\n🔎 DETALHE DO BUDGET (Mês {idx_m12 + 1}):")
mkt_m12 = df_base['Marketing'].iloc[idx_m12]
print(f"  • Budget Total: {formatar_moeda(mkt_m12)}")

canais_check = [
    ('Google', PREMISSAS['canal_google_pct']),
    ('YouTube', PREMISSAS['canal_youtube_pct']),
    ('Instagram', PREMISSAS['canal_instagram_pct']),
    ('Facebook', PREMISSAS['canal_facebook_pct'])
]

for canal, pct in canais_check:
    valor = mkt_m12 * pct
    print(f"  • {canal} ({pct*100:.0f}%): {formatar_moeda(valor)}")

print("✅ CELULA 10: Hardcoded removidos (cores, threshold orgânico)")


📊 ANÁLISE DE DEPENDÊNCIA ORGÂNICA
  • Dependência Média: 9.3% do crescimento vem do orgânico
  • Déficit de Meta (36m): 0 usuários precisaram vir do orgânico
  ✅ SAUDÁVEL: Budget de marketing suporta a maior parte do crescimento.
     O orgânico é um bônus para reduzir CAC, não uma necessidade vital.

🔎 DETALHE DO BUDGET (Mês 12):
  • Budget Total: R$ 4.907,00
  • Google (30%): R$ 1.472,10
  • YouTube (30%): R$ 1.472,10
  • Instagram (20%): R$ 981,40
  • Facebook (20%): R$ 981,40
✅ CELULA 10: Hardcoded removidos (cores, threshold orgânico)


In [10]:
# =========================================================================
# CELULA 10: PAINEL FINANCEIRO INTEGRADO (FLUXO + CAIXA)
# =========================================================================
from plotly.subplots import make_subplots

meses = df_base['Mes']

# Dados consolidados
custos_overhead = df_base['Custo_Escritorio'] + df_base['Custo_Adm_Terceiros'] + df_base['Custo_Contingencia']
custo_total_real = df_base['OPEX_Total'] + df_base['COGS_Total']
receita = df_base['Receita_Mensal']
caixa = df_base['Saldo_Caixa']

# Criar Figura com 2 Linhas (Subplots)
fig_fin = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True, # Alinhar eixo X
    vertical_spacing=0.1,
    subplot_titles=("<b>FLUXO MENSAL: Receita vs Custos Detalhados</b>", "<b>ESTOQUE DE CAIXA: Capacidade de Investimento</b>"),
    row_heights=[0.6, 0.4] # Gráfico de cima maior
)

# --- GRÁFICO 1 (CIMA): FLUXO MENSAL ---

# 1.1 Receita (Fundo Sombreado para dar contexto de "Teto")
fig_fin.add_trace(go.Scatter(
    x=meses, y=receita,
    name='Receita Total',
    mode='lines',
    line=dict(color='#10B981', width=0), # Linha invisível
    fill='tozeroy',
    fillcolor='rgba(16, 185, 129, 0.05)' # Verde bem clarinho
), row=1, col=1)

# 1.2 Barras de Custo (Empilhadas)
fig_fin.add_trace(go.Bar(x=meses, y=df_base['COGS_Total'], name='COGS (Var)', marker_color=PREMISSAS['cores']['cogs']), row=1, col=1)
fig_fin.add_trace(go.Bar(x=meses, y=custos_overhead, name='Overhead/Adm', marker_color='#94A3B8'), row=1, col=1)
fig_fin.add_trace(go.Bar(x=meses, y=df_base['Marketing'], name='Marketing', marker_color=PREMISSAS['cores']['marketing_custo']), row=1, col=1)
fig_fin.add_trace(go.Bar(x=meses, y=df_base['Infraestrutura'], name='Infra', marker_color=PREMISSAS['cores']['infra']), row=1, col=1)
fig_fin.add_trace(go.Bar(x=meses, y=df_base['Pessoal'], name='Pessoal', marker_color=PREMISSAS['cores']['pessoal']), row=1, col=1)

# 1.3 Linha de Lucro Líquido (A "Verdade" do mês)
fig_fin.add_trace(go.Scatter(
    x=meses, y=df_base['Lucro_Liquido'],
    name='Lucro Líquido',
    line=dict(color='#059669', width=3)
), row=1, col=1)

# --- GRÁFICO 2 (BAIXO): CAIXA ACUMULADO ---

# 2.1 Área de Caixa
fig_fin.add_trace(go.Scatter(
    x=meses, y=caixa,
    name='Saldo em Caixa',
    mode='lines',
    line=dict(color='#3B82F6', width=2),
    fill='tozeroy',
    fillcolor='rgba(59, 130, 246, 0.1)'
), row=2, col=1)

# 2.2 Linha de Segurança (Meta)
meta_caixa = PREMISSAS.get('meta_caixa_seguranca', 50000)
fig_fin.add_hline(y=meta_caixa, line_dash="dot", line_color="gray", annotation_text="Caixa Mínimo", row=2, col=1)

# --- ANOTAÇÕES E EVENTOS (Contexto entre os gráficos) ---

# Abertura Escritório
mes_office = df_base[df_base['Custo_Escritorio'] > 0]['Mes'].min() if any(df_base['Custo_Escritorio'] > 0) else None
if mes_office:
    fig_fin.add_vline(x=mes_office, line_width=1, line_dash="dash", line_color="gray")
    fig_fin.add_annotation(
        x=mes_office, y=1, yref="paper",
        text="🏢 Escritório", showarrow=False,
        bgcolor="#F3F4F6", font=dict(size=10)
    )

# Capex (Impacto no Caixa)
mes_capex = PREMISSAS['mes_capex_recorrente']
if mes_capex <= len(df_base):
    val_capex = PREMISSAS['capex_recorrente_mes_24']
    # Seta no gráfico de caixa mostrando a queda
    caixa_momento = caixa.iloc[mes_capex-1]
    fig_fin.add_annotation(
        x=mes_capex, y=caixa_momento,
        text=f"🔻 CAPEX<br>{formatar_moeda(val_capex)}",
        showarrow=True, arrowhead=2, arrowcolor='#DC2626', ax=0, ay=-40,
        row=2, col=1
    )

# Layout Final
fig_fin.update_layout(
    title="<b>Painel Financeiro Integrado: Fluxo Econômico & Solvência de Caixa</b>",
    barmode='stack',
    template='plotly_white',
    height=800, # Mais alto para caber os dois
    hovermode='x unified',
    showlegend=True
)

fig_fin.update_yaxes(title_text="R$ (Mensal)", row=1, col=1)
fig_fin.update_yaxes(title_text="R$ (Acumulado)", row=2, col=1)

fig_fin.show()

# Relatório Rápido
caixa_final = caixa.iloc[-1]
custo_final = custo_total_real.iloc[-1]
meses_runway = caixa_final / custo_final if custo_final > 0 else 99

print(f"\n💰 ANÁLISE DE SOLVÊNCIA (M36):")
print(f"  • Caixa Acumulado: {formatar_moeda(caixa_final)}")
print(f"  • Custo Mensal Atual: {formatar_moeda(custo_final)}")
print(f"  • Runway (Sobrevivência sem receita): {meses_runway:.1f} meses")
if caixa_final > val_capex * 2:
    print("  ✅ CAIXA ROBUSTO: Permite novos investimentos ou contratações.")
else:
    print("  ⚠️ CAIXA APERTADO: Cuidado com novos custos fixos.")


💰 ANÁLISE DE SOLVÊNCIA (M36):
  • Caixa Acumulado: R$ 116.401,14
  • Custo Mensal Atual: R$ 90.709,82
  • Runway (Sobrevivência sem receita): 1.3 meses
  ✅ CAIXA ROBUSTO: Permite novos investimentos ou contratações.


In [11]:
# ============================================================================
# CELULA 11: SANKEY DIAGRAM (FLUXO DO DINHEIRO M36)
# ============================================================================
import plotly.graph_objects as go

# 1. Preparar Dados do Mês 36
row = df_base.iloc[-1]

# Definir Nós (Labels)
labels = [
    "Receita Total",           # 0
    "Custos Variáveis (COGS)", # 1
    "Margem de Contribuição",  # 2
    "Marketing & Growth",      # 3
    "Pessoal & Equipe",        # 4
    "Infra & Tecnologia",      # 5
    "Admin & Escritório",      # 6
    "Lucro Líquido"            # 7
]

# Definir Cores dos Nós (Usando premissas)
colors = [
    PREMISSAS['cores']['revenue'],   # Receita
    PREMISSAS['cores']['cogs'],      # COGS
    PREMISSAS['cores']['primary'],   # Margem
    PREMISSAS['cores']['marketing_custo'],
    PREMISSAS['cores']['pessoal'],
    PREMISSAS['cores']['infra'],
    '#94A3B8',                       # Admin (Cinza)
    PREMISSAS['cores']['profit']     # Lucro
]

# Definir Fluxos (Source -> Target)
# Índices baseados na lista 'labels' acima
sources = [0, 0, 2, 2, 2, 2, 2] 
targets = [1, 2, 3, 4, 5, 6, 7] 

# Valores dos fluxos
values = [
    row['COGS_Total'],           # Receita -> COGS
    row['Lucro_Bruto'],          # Receita -> Margem (O que sobra)
    
    # Da Margem para os OPEX
    row['Marketing'] + row['Comissoes_Afiliados'], # -> Marketing
    row['Pessoal'],                                # -> Pessoal
    row['Infraestrutura'],                         # -> Infra
    row['Custo_Escritorio'] + row['Custo_Adm_Terceiros'] + row['Custo_Contingencia'], # -> Admin
    
    # O que sobra é Lucro
    row['Lucro_Liquido']
]

# Criar Sankey
fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=colors,
        hovertemplate='%{label}<br>R$ %{value:,.2f}<extra></extra>' # Formatação no hover
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=['rgba(255, 86, 48, 0.2)',  # COGS (Vermelho claro)
               'rgba(54, 179, 126, 0.2)', # Margem (Verde claro)
               'rgba(245, 158, 11, 0.2)', # Mkt
               'rgba(59, 130, 246, 0.2)', # Pessoal
               'rgba(100, 116, 139, 0.2)',# Infra
               'rgba(148, 163, 184, 0.2)',# Admin
               'rgba(0, 82, 204, 0.4)']   # Lucro (Azul forte)
    )
)])

fig_sankey.update_layout(
    title_text=f"<b>Fluxo do Dinheiro (M36): R$ {formatar_moeda(row['Receita_Mensal'])}</b><br><sup>Como cada Real da receita é distribuído</sup>",
    font_size=12,
    template='plotly_white',
    height=500
)

fig_sankey.show()

# Texto Explicativo
print(f"\n🌊 ANÁLISE DE FLUXO (Sankey):")
print(f"  O gráfico acima mostra o caminho do dinheiro.")
print(f"  1. Da Receita Total, subtrai-se o COGS para chegar na Margem.")
print(f"  2. A Margem paga todos os departamentos (Mkt, Pessoal, Infra, Admin).")
print(f"  3. O que resta na ponta direita é o seu Lucro Líquido Real ({formatar_moeda(row['Lucro_Liquido'])}).")


🌊 ANÁLISE DE FLUXO (Sankey):
  O gráfico acima mostra o caminho do dinheiro.
  1. Da Receita Total, subtrai-se o COGS para chegar na Margem.
  2. A Margem paga todos os departamentos (Mkt, Pessoal, Infra, Admin).
  3. O que resta na ponta direita é o seu Lucro Líquido Real (R$ 22.112,06).


In [38]:
# ============================================================================
# CELULA 15: ANÁLISE GRANULAR DE CUSTOS (DEEP DIVE)
# ============================================================================
# Visualização hierárquica e temporal de onde o dinheiro é gasto.
# ============================================================================

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. GARANTIR DADOS ATUALIZADOS
# Rodamos a simulação base para garantir que temos as novas colunas (Escritório, Adm, etc)
try:
    df_analise = simular_projecao(PREMISSAS)
except Exception as e:
    print(f"⚠️ Erro ao rodar simulação base: {e}. Usando dados anteriores se existirem.")
    df_analise = df_base # Fallback

# Pegamos o Mês 36 (Estabilizado) para a foto estática
m36 = df_analise.iloc[-1] 

# ============================================================================
# VISÃO 1: SUNBURST CHART (A "Pizza" Hierárquica)
# ============================================================================
# Aqui quebramos artificialmente os agrupamentos para mostrar o detalhe
# (Internet, Aluguel, Softwares, etc) baseados nas premissas.

# Estrutura dos dados para o Sunburst
data_sunburst = []

# --- GRUPO 1: COGS (Custos Variáveis) ---
data_sunburst.append(dict(ids="Custos", parent="", value=0, label="Custos Totais"))
data_sunburst.append(dict(ids="COGS", parent="Custos", value=m36['COGS_Total'], label="COGS (Variável)"))

# Detalhes do COGS
data_sunburst.append(dict(ids="Impostos", parent="COGS", value=m36['Impostos'], label="Impostos (NF)"))
data_sunburst.append(dict(ids="Taxas Pagto", parent="COGS", value=m36['Taxas_Pagamento'], label="Gateway (Taxas)"))
data_sunburst.append(dict(ids="Custo IA/API", parent="COGS", value=m36['Custo_IA'], label="APIs & IA Token"))

# --- GRUPO 2: OPEX (Custos Fixos) ---
data_sunburst.append(dict(ids="OPEX", parent="Custos", value=m36['OPEX_Total'], label="OPEX (Fixo)"))

# 2.1 Pessoal (Detalhando por cargo estimado)
# Recalculando estimativa proporcional para visualização
folha_total = m36['Pessoal']
# Estimativas baseadas nas premissas para 'quebrar' o valor total visualmente
salario_dev = PREMISSAS['salario_dev_senior'] * (1 + PREMISSAS['encargos'])
salario_cs = PREMISSAS['salario_cs'] * (1 + PREMISSAS['encargos'])
salario_founder = PREMISSAS['salario_fundador'] * (1 + PREMISSAS['encargos'])
# Assumindo proporção simples se o total não bater exato
total_teorico = salario_dev + salario_cs + salario_founder
pct_dev = salario_dev / total_teorico
pct_cs = salario_cs / total_teorico
pct_founder = salario_founder / total_teorico

data_sunburst.append(dict(ids="Equipe", parent="OPEX", value=folha_total, label="Equipe & RH"))
data_sunburst.append(dict(ids="Tech Team", parent="Equipe", value=folha_total*pct_dev, label="Devs (Tech)"))
data_sunburst.append(dict(ids="Suporte", parent="Equipe", value=folha_total*pct_cs, label="CS/Suporte"))
data_sunburst.append(dict(ids="C-Level", parent="Equipe", value=folha_total*pct_founder, label="Diretoria"))

# 2.2 Infraestrutura & Tech
infra_total = m36['Infraestrutura']
# Quebra estimada
data_sunburst.append(dict(ids="Tecnologia", parent="OPEX", value=infra_total, label="Infra & Tech"))
data_sunburst.append(dict(ids="Servidores", parent="Tecnologia", value=infra_total*0.7, label="Cloud (AWS/VPS)"))
data_sunburst.append(dict(ids="Softwares", parent="Tecnologia", value=infra_total*0.3, label="SaaS & Ferramentas"))

# 2.3 Escritório (O detalhe que você pediu: Internet, Luz, Aluguel)
office_total = m36['Custo_Escritorio']
if office_total > 0:
    # Recuperando valores das premissas ou usando defaults proporcionais
    aluguel = PREMISSAS.get('custo_aluguel_condominio', 3500)
    internet = PREMISSAS.get('custo_internet_redundante', 600)
    limpeza = PREMISSAS.get('custo_utilidades_limpeza', 800)
    total_itens = aluguel + internet + limpeza
    
    # Ajustar para bater com o valor da simulação
    fator = office_total / total_itens
    
    data_sunburst.append(dict(ids="Escritório", parent="OPEX", value=office_total, label="Instalações"))
    data_sunburst.append(dict(ids="Aluguel", parent="Escritório", value=aluguel*fator, label="Aluguel/Condo"))
    data_sunburst.append(dict(ids="Conectividade", parent="Escritório", value=internet*fator, label="Internet/Rede"))
    data_sunburst.append(dict(ids="Utilities", parent="Escritório", value=limpeza*fator, label="Luz/Limpeza/Copa"))

# 2.4 Marketing
mkt_total = m36['Marketing'] + m36['Comissoes_Afiliados']
data_sunburst.append(dict(ids="Growth", parent="OPEX", value=mkt_total, label="Growth"))
data_sunburst.append(dict(ids="Ads Pago", parent="Growth", value=m36['Marketing'], label="Mídia Paga (Ads)"))
data_sunburst.append(dict(ids="Afiliados", parent="Growth", value=m36['Comissoes_Afiliados'], label="Comissões"))

# 2.5 Admin & Outros
adm_total = m36['Custo_Adm_Terceiros'] + m36['Custo_Contingencia']
data_sunburst.append(dict(ids="Admin", parent="OPEX", value=adm_total, label="Admin & Geral"))
data_sunburst.append(dict(ids="Contabilidade", parent="Admin", value=m36['Custo_Adm_Terceiros'], label="Contábil/Jurídico"))
data_sunburst.append(dict(ids="Reserva", parent="Admin", value=m36['Custo_Contingencia'], label="Imprevistos"))

# Gerar Gráfico Sunburst
df_sb = pd.DataFrame(data_sunburst)
fig_sun = go.Figure(go.Sunburst(
    ids=df_sb.ids,
    labels=df_sb.label,
    parents=df_sb.parent,
    values=df_sb.value,
    branchvalues="total",
    hovertemplate='<b>%{label}</b><br>Custo: R$ %{value:,.2f}<br>%{percentRoot:.1%} do Total<extra></extra>',
    marker=dict(colorscale='Tealgrn')
))

fig_sun.update_layout(
    title="<b>DECOMPOSIÇÃO DE CUSTOS (DRILL-DOWN) - MÊS 36</b><br><sup>Clique no centro para voltar, clique nas fatias para detalhar.</sup>",
    height=600,
    margin = dict(t=50, l=0, r=0, b=0),
    template='plotly_white'
)
fig_sun.show()

# ============================================================================
# VISÃO 2: ÁREA EMPILHADA (EVOLUÇÃO DOS CUSTOS)
# ============================================================================
fig_stack = go.Figure()

colunas_stack = {
    'Custo_IA': 'Custos IA/Servidor',
    'Impostos': 'Impostos',
    'Marketing': 'Ads & Marketing',
    'Pessoal': 'Salários & Encargos',
    'Infraestrutura': 'Infra & Tech',
    'Custo_Escritorio': 'Escritório Físico',
    'Custo_Adm_Terceiros': 'Admin/Legal'
}

# Adicionar cada categoria
for col, nome in colunas_stack.items():
    if df_analise[col].sum() > 0: # Só adiciona se tiver valor
        fig_stack.add_trace(go.Scatter(
            x=df_analise['Mes'], y=df_analise[col],
            mode='lines',
            name=nome,
            stackgroup='one', # Isso cria o empilhamento
            groupnorm=None    # Use 'percent' aqui se quiser ver % em vez de valor absoluto
        ))

fig_stack.update_layout(
    title=f"<b>EVOLUÇÃO DA ESTRUTURA DE CUSTOS ({PREMISSAS['meses_projecao']} MESES)</b><br><sup>Como o gasto muda de 'Marketing' para 'Estrutura' conforme a empresa escala</sup>",
    xaxis_title="Mês de Operação",
    yaxis_title="Custo Mensal (R$)",
    template='plotly_white',
    height=500,
    hovermode='x unified'
)
fig_stack.update_yaxes(tickformat=",.0f")
fig_stack.show()

# ============================================================================
# TABELA DETALHADA "TIPO EXCEL" (P&L MENSAL)
# ============================================================================
cols_export = [
    'Mes', 'Receita_Mensal', 
    'Custo_IA', 'Taxas_Pagamento', 'Impostos', 'COGS_Total',
    'Marketing', 'Pessoal', 'Infraestrutura', 'Custo_Escritorio', 'Custo_Adm_Terceiros', 
    'OPEX_Total', 'EBITDA', 'Lucro_Liquido'
]

# Selecionar meses chaves (Início, Meio, Fim)
meses_selecionados = [1, 3, 6, 12, 18, 24, 30, 36]
# Filtrar apenas os que existem na projeção
meses_reais = [m for m in meses_selecionados if m <= PREMISSAS['meses_projecao']]

df_table = df_analise.loc[df_analise['Mes'].isin(meses_reais), cols_export].copy()

# Formatar para exibição
print("\n" + "="*100)
print("📋 DEMONSTRATIVO DE RESULTADOS DETALHADO (DRE ANALÍTICO)")
print("Este detalhamento mostra itens como 'IA', 'Escritório' e 'Admin' separadamente.")
print("="*100)

for col in cols_export[1:]: # Pula coluna 'Mes'
    df_table[col] = df_table[col].apply(lambda x: f"{x:,.2f}")

# Renomear para ficar bonito
rename_map = {
    'Receita_Mensal': 'Receita Bruta',
    'Custo_IA': 'Custo API/IA',
    'Taxas_Pagamento': 'Gateway (%)',
    'Impostos': 'Impostos (NF)',
    'Custo_Escritorio': 'Escritório/Luz/Net',
    'Custo_Adm_Terceiros': 'Contábil/Sistemas',
    'Infraestrutura': 'Infra Cloud'
}
df_table.rename(columns=rename_map, inplace=True)

print(df_table.to_markdown(index=False))

# Insights Automáticos
print("\n💡 INSIGHTS DE CUSTOS:")
cogs_pct = (m36['COGS_Total'] / m36['Receita_Mensal']) * 100
opex_pct = (m36['OPEX_Total'] / m36['Receita_Mensal']) * 100
marketing_share = (m36['Marketing'] / m36['OPEX_Total']) * 100

print(f"1. PESO DO PRODUTO (COGS): {cogs_pct:.1f}% da receita vai para entregar o serviço (IA, Taxas, Impostos).")
print(f"2. PESO DA OPERAÇÃO (OPEX): {opex_pct:.1f}% da receita paga a estrutura fixa.")
print(f"3. ENGINE DE CRESCIMENTO: Marketing representa {marketing_share:.1f}% de todos os custos fixos no mês 36.")
if m36['Custo_Escritorio'] > 0:
    print(f"4. ESTRUTURA FÍSICA: O escritório (Aluguel+Internet+Limpeza) custa R$ {formatar_moeda(m36['Custo_Escritorio'])}/mês.")
else:
    print(f"4. ESTRUTURA: Operação 100% Remota (Custo de escritório = 0).")


📋 DEMONSTRATIVO DE RESULTADOS DETALHADO (DRE ANALÍTICO)
Este detalhamento mostra itens como 'IA', 'Escritório' e 'Admin' separadamente.
|   Mes | Receita Bruta   | Custo API/IA   | Gateway (%)   | Impostos (NF)   | COGS_Total   | Marketing   | Pessoal   | Infra Cloud   | Escritório/Luz/Net   |   Contábil/Sistemas | OPEX_Total   | EBITDA    | Lucro_Liquido   |
|------:|:----------------|:---------------|:--------------|:----------------|:-------------|:------------|:----------|:--------------|:---------------------|--------------------:|:-------------|:----------|:----------------|
|     1 | 2,435.86        | 150.70         | 85.26         | 146.15          | 382.11       | 3,000.00    | 0.00      | 964.00        | 0.00                 |                 800 | 5,014.88     | -2,961.12 | -2,961.12       |
|     3 | 3,946.39        | 244.15         | 138.12        | 236.78          | 619.06       | 3,000.00    | 0.00      | 964.00        | 0.00                 |                 800 | 5,14

In [12]:
# ============================================================================
# CELULA 12
# ============================================================================
# MONTE CARLO SIMULATION (COM VARIÁVEIS DO FUNIL)
# ============================================================================

def monte_carlo_simulation(premissas, n_simulacoes=None):
    """
    Monte Carlo com variáveis do funil incluídas.
    
    Variáveis estocásticas (10 + 3 do funil = 13 total):
    1. churn_base
    2. custo_ia_por_usuario
    3. marketing_perc_receita
    4. infra_fase2
    5. taxa_pagamento
    6. taxa_visitante_para_trial (NOVO)
    7. taxa_trial_para_pagante (NOVO)
    8. crescimento_trafego_mes_1_6 (NOVO)
    9. mix_lite (NOVO - mix de planos)
    10. meses_aporte (chance de atraso definida em premissas)
    """
    # Usa valor padrão das premissas se não informado
    n_sims = n_simulacoes if n_simulacoes else premissas['mc_n_simulacoes']
    
    resultados_mrr = []
    resultados_caixa = []
    resultados_usuarios = []
    resultados_breakeven = []
    resultados_retorno = []
    resultados_ltv_cac = []
    
    print(f"🎲 Executando {n_sims} simulações Monte Carlo...")
    for i in range(n_sims):
        if (i + 1) % 500 == 0:
            print(f"  -> Simulação {i+1}/{n_sims}")
        
        # Gerar variações estocásticas baseadas nas PREMISSAS
        p = premissas # Alias para facilitar leitura
        
        variacao = {
            # 1. Churn: Distribuição Normal centrada na premissa
            'churn_base': np.clip(np.random.normal(p['churn_base'], p['churn_base'] * p['mc_std_churn']), 0.01, 0.20),
            
            # 2. Custo IA: Variação discreta (Cenários)
            'custo_ia_por_usuario': np.random.choice([
                p['custo_ia_por_usuario'] * p['mc_var_ia_otimista'], # Otimista
                p['custo_ia_por_usuario'],                           # Base
                p['custo_ia_por_usuario'] * p['mc_var_ia_pessimista']  # Pessimista
            ]),
            
            # 3. Marketing: Normal centrada na premissa
            'marketing_perc_receita': np.clip(np.random.normal(p['marketing_perc_receita'], p['marketing_perc_receita'] * p['mc_std_marketing']), 0.10, 0.30),
            
            # 4. Infra Fase 2: Normal centrada na premissa
            'infra_fase2': max(1000, np.random.normal(p['infra_fase2'], p['infra_fase2'] * p['mc_std_infra'])),
            
            # 5. Taxa Pagamento: Normal centrada na premissa (variação absoluta pequena)
            'taxa_pagamento': np.clip(np.random.normal(p['taxa_pagamento'], p['mc_std_taxa_pag']), 0.01, 0.06),
            
            # 6. Taxa Visitante -> Trial
            'taxa_visitante_para_trial': np.clip(np.random.normal(p['taxa_visitante_para_trial'], p['taxa_visitante_para_trial'] * p['mc_std_vis_trial']), 0.01, 0.20),
            
            # 7. Taxa Trial -> Pagante
            'taxa_trial_para_pagante': np.clip(np.random.normal(p['taxa_trial_para_pagante'], p['taxa_trial_para_pagante'] * p['mc_std_trial_pag']), 0.01, 0.30),
            
            # 8. Crescimento Tráfego Inicial
            'crescimento_trafego_mes_1_6': np.clip(np.random.normal(p['crescimento_trafego_mes_1_6'], p['crescimento_trafego_mes_1_6'] * p['mc_std_cresc_traf']), 0.05, 0.50),
            
            # 9. Mix Lite (Plano de entrada) - Variação absoluta
            'mix_lite': np.clip(np.random.normal(p['mix_lite'], p['mc_std_mix_lite']), 0.30, 0.80),
        }
        
        # Ajustar mix_trader e mix_pro para somar 1 (mantendo proporção relativa original)
        prop_trader = p['mix_trader'] / (p['mix_trader'] + p['mix_pro'])
        
        resto = 1.0 - variacao['mix_lite']
        variacao['mix_trader'] = resto * prop_trader
        variacao['mix_pro'] = resto * (1 - prop_trader)
        
        # Aporte delay (Chance de atraso de 1 mês)
        if np.random.random() < p['mc_prob_atraso_aporte']:
            variacao['meses_aporte'] = max(0, p['meses_aporte'] - 1)
        
        # Simular
        df = simular_projecao(p, variacao_params=variacao, seed=i)
        
        # Coletar resultados
        resultados_mrr.append(df['MRR'].iloc[-1])
        resultados_caixa.append(df['Saldo_Caixa'].iloc[-1])
        resultados_usuarios.append(df['Usuarios_Finais'].iloc[-1])
        resultados_retorno.append(df['Dist_Investidor'].sum())
        resultados_ltv_cac.append(df['LTV_CAC_Ratio'].iloc[-1])
        
        # Break-even (se não atingir, definir como meses_projecao + 1)
        be = df[df['Lucro_Liquido'] > 0]['Mes'].min() if any(df['Lucro_Liquido'] > 0) else (p['meses_projecao'] + 1)
        resultados_breakeven.append(be)
    
    resultados = {
        'MRR_Final': np.array(resultados_mrr),
        'Caixa_Final': np.array(resultados_caixa),
        'Usuarios_Final': np.array(resultados_usuarios),
        'Break_Even_Mes': np.array(resultados_breakeven),
        'Retorno_Investidor': np.array(resultados_retorno),
        'LTV_CAC_Ratio': np.array(resultados_ltv_cac)
    }
    
    return resultados

# Executar Monte Carlo
# Usa o número de simulações definido em PREMISSAS
mc_results = monte_carlo_simulation(PREMISSAS)

# Estatísticas
print("\n" + "="*60)
print(f"📊 RESULTADOS MONTE CARLO ({PREMISSAS['mc_n_simulacoes']} simulações)")
print("="*60)

for metrica, dados in mc_results.items():
    p10 = np.percentile(dados, 10)
    p50 = np.percentile(dados, 50)
    p90 = np.percentile(dados, 90)
    
    if 'MRR' in metrica or 'Caixa' in metrica or 'Retorno' in metrica:
        print(f"\n{metrica}:")
        print(f"  P10 (10% de probabilidade): {formatar_moeda(p10)}")
        print(f"  P50 (50% de probabilidade - mediana): {formatar_moeda(p50)}")
        print(f"  P90 (90% de probabilidade): {formatar_moeda(p90)}")
    elif 'Mes' in metrica:
        print(f"\n{metrica}:")
        print(f"  P10 (10% de probabilidade): Mês {int(p10)}")
        print(f"  P50 (50% de probabilidade): Mês {int(p50)}")
        print(f"  P90 (90% de probabilidade): Mês {int(p90)}")
    else:
        print(f"\n{metrica}:")
        print(f"  P10: {p10:.1f}")
        print(f"  P50: {p50:.1f}")
        print(f"  P90: {p90:.1f}")

# Análise de risco
prob_caixa_negativo = (mc_results['Caixa_Final'] < 0).mean() * 100
prob_retorno_positivo = (mc_results['Retorno_Investidor'] > 0).mean() * 100
# Usando benchmark dinâmico para análise de LTV
thresh_ltv = PREMISSAS['benchmark_ltv_cac_atencao']
prob_ltv_saudavel = (mc_results['LTV_CAC_Ratio'] > thresh_ltv).mean() * 100

print("\n" + "="*60)
print("⚠️  ANÁLISE DE RISCO")
print("="*60)
print(f"  • Probabilidade de caixa negativo (M{PREMISSAS['meses_projecao']}): {prob_caixa_negativo:.1f}%")
print(f"  • Probabilidade de retorno positivo: {prob_retorno_positivo:.1f}%")
print(f"  • Probabilidade de Relação LTV/CAC > {thresh_ltv}x: {prob_ltv_saudavel:.1f}%")
print("✅ CELULA 11: Hardcoded removidos (parâmetros de simulação)")

🎲 Executando 15000 simulações Monte Carlo...
  -> Simulação 500/15000
  -> Simulação 1000/15000
  -> Simulação 1500/15000
  -> Simulação 2000/15000
  -> Simulação 2500/15000
  -> Simulação 3000/15000
  -> Simulação 3500/15000
  -> Simulação 4000/15000
  -> Simulação 4500/15000
  -> Simulação 5000/15000
  -> Simulação 5500/15000
  -> Simulação 6000/15000
  -> Simulação 6500/15000
  -> Simulação 7000/15000
  -> Simulação 7500/15000
  -> Simulação 8000/15000
  -> Simulação 8500/15000
  -> Simulação 9000/15000
  -> Simulação 9500/15000
  -> Simulação 10000/15000
  -> Simulação 10500/15000
  -> Simulação 11000/15000
  -> Simulação 11500/15000
  -> Simulação 12000/15000
  -> Simulação 12500/15000
  -> Simulação 13000/15000
  -> Simulação 13500/15000
  -> Simulação 14000/15000
  -> Simulação 14500/15000
  -> Simulação 15000/15000

📊 RESULTADOS MONTE CARLO (15000 simulações)

MRR_Final:
  P10 (10% de probabilidade): R$ 11.686,08
  P50 (50% de probabilidade - mediana): R$ 75.307,12
  P90 (90% d

In [13]:
# ============================================================================
# CELULA 13: ANÁLISE DE RISCO - CAIXA FINAL (VISUALIZAÇÃO CORRIGIDA & INTELIGENTE)
# ============================================================================
import plotly.graph_objects as go
import numpy as np

# Dados da simulação Monte Carlo (já calculados na Célula 11)
dados_caixa = mc_results['Caixa_Final']

# Estatísticas Chave
p10 = np.percentile(dados_caixa, 10)
p50 = np.percentile(dados_caixa, 50) # Mediana
p90 = np.percentile(dados_caixa, 90)
media = np.mean(dados_caixa)

# Configuração do Gráfico
fig_mc = go.Figure()

# Histograma (Distribuição de Frequência)
fig_mc.add_trace(go.Histogram(
    x=dados_caixa, 
    nbinsx=50, 
    marker_color=PREMISSAS['cores']['pessoal'], 
    opacity=0.7, 
    name="Cenários"
))

# --- LINHAS VERTICAIS (MARCADORES) ---
# P10 (Cenário Pessimista)
fig_mc.add_vline(x=p10, line_dash="dash", line_color=PREMISSAS['cores']['danger'], line_width=2)
# P50 (Cenário Provável/Mediana)
fig_mc.add_vline(x=p50, line_dash="solid", line_color=PREMISSAS['cores']['linha_referencia'], line_width=3)
# P90 (Cenário Otimista)
fig_mc.add_vline(x=p90, line_dash="dash", line_color=PREMISSAS['cores']['success'], line_width=2)
# Zero (Quebra de Caixa)
fig_mc.add_vline(x=0, line_color="black", line_width=1)

# --- ANOTAÇÕES INTELIGENTES (POSICIONAMENTO DINÂMICO) ---
# Ajustamos a posição Y (altura) para evitar sobreposição das caixas de texto.
# O eixo X agora segue o valor exato (p10, p50, p90) em vez de posição fixa na tela.

# P10 - Esquerda/Baixo
fig_mc.add_annotation(
    x=p10, y=0.85, yref="paper", # Posição relativa à altura do gráfico
    text=f"<b>P10 (Pessimista)</b><br>{formatar_moeda(p10)}",
    showarrow=True, arrowhead=1, ax=-40, ay=-40, # Seta apontando para a linha
    bgcolor="white", bordercolor=PREMISSAS['cores']['danger'], borderwidth=1
)

# P50 - Centro/Topo (Destaque)
fig_mc.add_annotation(
    x=p50, y=1.05, yref="paper",
    text=f"<b>P50 (Mediana)</b><br>{formatar_moeda(p50)}",
    showarrow=True, arrowhead=1, ax=0, ay=-40,
    bgcolor="white", bordercolor=PREMISSAS['cores']['linha_referencia'], borderwidth=2
)

# P90 - Direita/Meio
fig_mc.add_annotation(
    x=p90, y=0.60, yref="paper",
    text=f"<b>P90 (Otimista)</b><br>{formatar_moeda(p90)}",
    showarrow=True, arrowhead=1, ax=40, ay=-40,
    bgcolor="white", bordercolor=PREMISSAS['cores']['success'], borderwidth=1
)

# Risco de Quebra (Apenas se houver risco real > 1%)
prob_quebra = (dados_caixa < 0).mean() * 100
if prob_quebra > 1.0:
    fig_mc.add_annotation(
        x=0, y=0.2, yref="paper",
        text=f"RISCO DE QUEBRA<br>{prob_quebra:.1f}% dos cenários",
        showarrow=True, arrowhead=2, ax=-60, ay=0,
        bgcolor="#FEF2F2", bordercolor="red", font=dict(color="red")
    )

fig_mc.update_layout(
    title=dict(
        text=f"<b>Distribuição de Probabilidade: Caixa Final (M{PREMISSAS['meses_projecao']})</b><br><sup>Simulação de {len(dados_caixa)} cenários de Monte Carlo</sup>",
        font=dict(size=20)
    ),
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    xaxis=dict(title="Saldo em Caixa (R$)", gridcolor=PREMISSAS['cores']['grid']),
    yaxis=dict(title="Frequência (Cenários)", showgrid=False),
    bargap=0.1,
    margin=dict(t=120) # Margem superior maior para caber a anotação P50
)

fig_mc.show()

# --- ANÁLISE DE INSIGHTS INTELIGENTE (TEXTO DINÂMICO) ---
print(f"\n📊 ANÁLISE DE RISCO E LIQUIDEZ (Caixa M{PREMISSAS['meses_projecao']}):")

# 1. Análise de Assimetria (Para onde o gráfico "pende")
if p50 < media:
    tipo_assimetria = "POSITIVA (Cauda à Direita)"
    explicacao = "A maioria dos resultados concentra-se em valores moderados, mas existem cenários de sucesso explosivo (outliers positivos) que puxam a média para cima."
else:
    tipo_assimetria = "NEGATIVA (Cauda à Esquerda)"
    explicacao = "Há uma tendência de concentração em valores mais altos, mas o risco de resultados muito ruins puxa a média para baixo."

print(f"  • Perfil da Distribuição: {tipo_assimetria}")
print(f"    -> {explicacao}")

# 2. Análise de Risco de Quebra
print(f"\n📉 RISCO DE RUÍNA (Caixa Negativo): {prob_quebra:.1f}%")
if prob_quebra > 20:
    print("    🔴 CRÍTICO: Risco muito alto. Necessário aumentar aporte inicial ou reduzir custos fixos.")
elif prob_quebra > 5:
    print("    🟡 MODERADO: Existe risco real. Tenha um plano de contingência para novos aportes.")
else:
    print("    🟢 BAIXO: O modelo é robusto e se sustenta na grande maioria dos cenários.")

# 3. Análise de Dispersão (Volatilidade do Resultado)
dispersao = (p90 - p10)
print(f"\n↔️ INTERVALO DE INCERTEZA (P90 - P10): {formatar_moeda(dispersao)}")
print(f"    Isso significa que o resultado final pode variar em {formatar_moeda(dispersao)} dependendo das condições de mercado.")


📊 ANÁLISE DE RISCO E LIQUIDEZ (Caixa M36):
  • Perfil da Distribuição: POSITIVA (Cauda à Direita)
    -> A maioria dos resultados concentra-se em valores moderados, mas existem cenários de sucesso explosivo (outliers positivos) que puxam a média para cima.

📉 RISCO DE RUÍNA (Caixa Negativo): 7.7%
    🟡 MODERADO: Existe risco real. Tenha um plano de contingência para novos aportes.

↔️ INTERVALO DE INCERTEZA (P90 - P10): R$ 310.772,05
    Isso significa que o resultado final pode variar em R$ 310.772,05 dependendo das condições de mercado.


In [34]:
# ============================================================================
# CELULA 14
# ============================================================================
# MRR - DUPLA VISÃO CORRIGIDA (Zoom + Completa)
# ============================================================================

from scipy.stats import gaussian_kde
from plotly.subplots import make_subplots

p10_mrr = np.percentile(mc_results['MRR_Final'], 10)
p50_mrr = np.percentile(mc_results['MRR_Final'], 50)
p90_mrr = np.percentile(mc_results['MRR_Final'], 90)

# Ranges definidos nas premissas
zoom_max = PREMISSAS['layout_mrr_zoom_range']
full_max = PREMISSAS['layout_mrr_full_range']

# Calcular KDE
try:
    kde = gaussian_kde(mc_results['MRR_Final'])
    x_zoom = np.linspace(0, zoom_max, 500)
    x_full = np.linspace(0, full_max, 500)
    y_zoom = kde(x_zoom)
    y_full = kde(x_full)
except Exception as e:
    # Fallback caso dados sejam constantes
    print(f"⚠️ Aviso: Dados constantes, KDE ignorado. ({e})")
    x_zoom, x_full, y_zoom, y_full = [0], [0], [0], [0]

# Criar subplot 1x2
fig_mrr = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'<b>Zoom: Cenário Base (0-{int(zoom_max/1000)}k)</b>',
        f'<b>Visão Completa: Incluindo Otimista (0-{int(full_max/1000)}k)</b>'
    ),
    horizontal_spacing=0.12,
)

# ===== GRÁFICO 1: ZOOM =====
# Área sombreada P10-P50 (ou até max zoom)
if p10_mrr <= zoom_max:
    fig_mrr.add_vrect(
    x0=p10_mrr, x1=min(p50_mrr, zoom_max),
    fillcolor=PREMISSAS['cores']['primary'], opacity=0.15, line_width=0,
    row=1, col=1
    )

# Curva KDE
fig_mrr.add_trace(go.Scatter(
    x=x_zoom, y=y_zoom,
    mode='lines',
    line=dict(color=PREMISSAS['cores']['primary'], width=3),
    fill='tozeroy',
    fillcolor='rgba(0, 82, 204, 0.2)',
    showlegend=False,
), row=1, col=1)    
    
# Linha P10 (sempre visível no zoom se dentro do range)
if p10_mrr <= zoom_max * 2:
    fig_mrr.add_vline(
    x=p10_mrr, line_dash="dash", line_color=PREMISSAS['cores']['danger'], line_width=2,
    row=1, col=1
    )
    
# Linha P50 (se couber no range)
if p50_mrr <= zoom_max * 2:
    fig_mrr.add_vline(
    x=p50_mrr, line_dash="solid", line_color=PREMISSAS['cores']['primary'], line_width=3,
    row=1, col=1
    )
    
# ===== GRÁFICO 2: COMPLETO =====
# Área sombreada P10-P90
fig_mrr.add_vrect(
    x0=p10_mrr, x1=min(p90_mrr, full_max),
    fillcolor=PREMISSAS['cores']['success'], opacity=0.15, line_width=0,
    row=1, col=2
)
    
# Curva KDE
fig_mrr.add_trace(go.Scatter(
    x=x_full, y=y_full,
    mode='lines',
    line=dict(color=PREMISSAS['cores']['success'], width=3),
    fill='tozeroy',
    fillcolor='rgba(34, 139, 34, 0.2)',
    showlegend=False
), row=1, col=2)
    
# Linha P10
fig_mrr.add_vline(
    x=p10_mrr, line_dash="dash", line_color=PREMISSAS['cores']['danger'], line_width=2,
    row=1, col=2
)
    
# Linha P50
fig_mrr.add_vline(
    x=p50_mrr, line_dash="solid", line_color=PREMISSAS['cores']['primary'], line_width=3,
    row=1, col=2
)
    
# Linha P90
if p90_mrr <= full_max:
    fig_mrr.add_vline(
        x=p90_mrr, line_dash="dash", line_color=PREMISSAS['cores']['success'], line_width=2,
        row=1, col=2
    )
    
# Layout
fig_mrr.update_xaxes(title_text='<b>MRR (R$)</b>', range=[0, zoom_max], tickformat=',.0f', row=1, col=1)
fig_mrr.update_xaxes(title_text='<b>MRR (R$)</b>', range=[0, full_max], tickformat=',.0f', row=1, col=2)
fig_mrr.update_yaxes(title_text='<b>Densidade</b>', showticklabels=False, row=1, col=1)
fig_mrr.update_yaxes(title_text='<b>Densidade</b>', showticklabels=False, row=1, col=2)
    
fig_mrr.update_layout(
    title={
    'text': f"<b>RECEITA MENSAL RECORRENTE (MRR) NO MÊS {PREMISSAS['meses_projecao']}</b><br>" +
    f"<sup>Distribuição de {PREMISSAS['mc_n_simulacoes']} cenários - Curva de Densidade (KDE)</sup>",
    'x': 0.5,
    'xanchor': 'center',
    'y': 0.95
    },
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    font=dict(size=11),
    margin=dict(t=140, b=80, l=80, r=80),
    showlegend=False,
    )
    
# Anotações (exibir sempre os 3 percentis)
fig_mrr.add_annotation(
    text=f"<b>P10 (Pessimista)</b><br>10% prob.<br>{formatar_moeda(p10_mrr)}",
    xref="paper", yref="paper", x=0.15, y=1.3,
    showarrow=False, font=dict(size=10), align='center',
    bgcolor='white', bordercolor=PREMISSAS['cores']['danger'], borderwidth=2, borderpad=6
)
    
fig_mrr.add_annotation(
    text=f"<b>P50 (Provável)</b><br>Mediana<br>{formatar_moeda(p50_mrr)}",
    xref="paper", yref="paper", x=0.5, y=1.3,
    showarrow=False, font=dict(size=11), align='center',
    bgcolor='white', bordercolor=PREMISSAS['cores']['primary'], borderwidth=3, borderpad=6
)
    
fig_mrr.add_annotation(
    text=f"<b>P90 (Otimista)</b><br>90% prob.<br>{formatar_moeda(p90_mrr)}",
    xref="paper", yref="paper", x=0.85, y=1.3,
    showarrow=False, font=dict(size=10), align='center',
    bgcolor='white', bordercolor=PREMISSAS['cores']['success'], borderwidth=2, borderpad=6
)
    
fig_mrr.show()

# ============================================================================
# ANÁLISE DE DISTRIBUIÇÃO E PICO PESSIMISTA
# ============================================================================
# Cálculos de contagem exata
n_total = PREMISSAS['mc_n_simulacoes']
count_pessimista = int(n_total * 0.10)  # Abaixo do P10
count_base = int(n_total * 0.80)        # Entre P10 e P90
count_otimista = int(n_total * 0.10)    # Acima do P90

# Encontrar o PICO (Moda) da distribuição
# Onde a curva é mais alta = Resultado mais frequente
pico_densidade_idx = np.argmax(y_full)
valor_pico_moda = x_full[pico_densidade_idx]

print(f"\n📊 ESTATÍSTICAS DA SIMULAÇÃO ({n_total} Cenários):")
print(f"  • Cenários Pessimistas (≤ P10): {count_pessimista} simulações resultaram em MRR ≤ {formatar_moeda(p10_mrr)}")
print(f"  • Cenários Base (P10 a P90): {count_base} simulações ficaram entre {formatar_moeda(p10_mrr)} e {formatar_moeda(p90_mrr)}")
print(f"  • Cenários Otimistas (≥ P90): {count_otimista} simulações superaram {formatar_moeda(p90_mrr)}")

print(f"\n📈 ANÁLISE DO PICO DE PROBABILIDADE (MODA):")
print(f"  • O resultado mais frequente (pico do gráfico) foi: {formatar_moeda(valor_pico_moda)}")
print(f"  • A Mediana (P50) foi: {formatar_moeda(p50_mrr)}")

print("\n🧐 POR QUE O PICO É MAIS BAIXO QUE A MÉDIA/MEDIANA?")
print("  Esta é uma distribuição 'Log-Normal' ou Assimétrica à Direita, típica de Startups:")
print("  1. A maioria dos negócios (o pico) cresce de forma moderada ou lenta.")
print("  2. Existe um limite inferior (você não pode ter receita negativa), o que 'empilha' resultados no início.")
print("  3. Porém, o lado do sucesso (direita) é infinito. Alguns poucos cenários explodem (viralizam),")
print("     puxando a Média e a Mediana para cima, longe do resultado 'típico' mais comum.")
print("  CONCLUSÃO: É normal ter mais chance de um resultado baixo do que um resultado estratosférico.")


📊 ESTATÍSTICAS DA SIMULAÇÃO (15000 Cenários):
  • Cenários Pessimistas (≤ P10): 1500 simulações resultaram em MRR ≤ R$ 11.686,08
  • Cenários Base (P10 a P90): 12000 simulações ficaram entre R$ 11.686,08 e R$ 178.582,61
  • Cenários Otimistas (≥ P90): 1500 simulações superaram R$ 178.582,61

📈 ANÁLISE DO PICO DE PROBABILIDADE (MODA):
  • O resultado mais frequente (pico do gráfico) foi: R$ 17.635,27
  • A Mediana (P50) foi: R$ 75.307,12

🧐 POR QUE O PICO É MAIS BAIXO QUE A MÉDIA/MEDIANA?
  Esta é uma distribuição 'Log-Normal' ou Assimétrica à Direita, típica de Startups:
  1. A maioria dos negócios (o pico) cresce de forma moderada ou lenta.
  2. Existe um limite inferior (você não pode ter receita negativa), o que 'empilha' resultados no início.
  3. Porém, o lado do sucesso (direita) é infinito. Alguns poucos cenários explodem (viralizam),
     puxando a Média e a Mediana para cima, longe do resultado 'típico' mais comum.
  CONCLUSÃO: É normal ter mais chance de um resultado baixo d

In [37]:
# ============================================================================
# CELULA 14.1 - CORREÇÃO DEFINITIVA (ZERO CENTS)
# ============================================================================
# MRR: EVOLUÇÃO MENSAL CORRIGIDA & DATA TABLE PROFISSIONAL
# ============================================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings

# Ignorar avisos matemáticos irrelevantes
warnings.filterwarnings('ignore')

# --- 1. CONFIGURAÇÕES E RECUPERAÇÃO DE DADOS ---
# Tenta recuperar as cores e prazos das premissas, ou usa padrão se der erro
meses = PREMISSAS.get('meses_projecao', 36)
receita_inicial = PREMISSAS.get('receita_inicial', 0) 
cor_pessimista = PREMISSAS.get('cores', {}).get('danger', '#FF4136')
cor_base = PREMISSAS.get('cores', {}).get('primary', '#0074D9')
cor_otimista = PREMISSAS.get('cores', {}).get('success', '#2ECC40')

# Se não tiver resultados anteriores, cria dados fictícios para teste (fallback)
try:
    sim_final_mrr = mc_results['MRR_Final']
except:
    sim_final_mrr = np.random.normal(50000, 15000, 1000)
    print("⚠️ AVISO: Usando dados de teste (simulação anterior não encontrada).")

# --- 2. NOVA LÓGICA DE RECONSTRUÇÃO (SEM CENTAVOS) ---
def reconstruir_caminho_inteligente(start_val, end_vals, n_steps, volatility=0.2):
    """
    Se o negócio começa do zero, usa crescimento linear/progressivo (venda a venda).
    Se o negócio já existe, usa crescimento logarítmico (composto).
    Isso evita o problema de 'R$ 0,05' nos primeiros meses.
    """
    n_sims = len(end_vals)
    dt = 1.0 / n_steps
    noise = np.random.normal(0, np.sqrt(dt), size=(n_sims, n_steps))
    W = np.cumsum(noise, axis=1)
    
    # Ponte Browniana (Bridge) - Força o ruído a zerar no final
    t_vals = np.arange(1, n_steps + 1) / n_steps
    W_bridge = W - np.outer(W[:, -1], t_vals)
    
    paths = np.zeros((n_sims, n_steps + 1))
    paths[:, 0] = start_val
    
    # DECISÃO INTELIGENTE DE CURVA
    if start_val < 100: # Se começa do Zero ou quase zero (< R$ 100)
        # MODELO: TRAÇÃO LINEAR (Ramping Up)
        # Assume que o time de vendas vai performando melhor mês a mês
        # Fórmula: Start + (End - Start) * (t + ruído)
        # Isso gera números reais desde o mês 1 (ex: R$ 500, R$ 1.200...)
        for i in range(n_steps):
            t = (i + 1) / n_steps
            # Adiciona uma leve curva quadrática (t^1.2) para simular que o começo é mais difícil
            progresso = t ** 1.2 
            
            caminho_base = start_val + (end_vals - start_val) * progresso
            # Aplica volatilidade sobre o progresso
            desvio = W_bridge[:, i] * volatility * (end_vals - start_val) 
            
            paths[:, i+1] = np.maximum(caminho_base + desvio, 0) # Garante que não fica negativo
            
    else:
        # MODELO: CRESCIMENTO COMPOSTO (Para empresas que já rodam)
        log_start = np.log(max(start_val, 1))
        log_ends = np.log(np.maximum(end_vals, 1))
        for i in range(n_steps):
            t = (i + 1) / n_steps
            log_path = log_start + (log_ends - log_start) * t + (W_bridge[:, i] * volatility * np.sqrt(n_steps))
            paths[:, i+1] = np.exp(log_path)
            
    return paths

print("🔄 Gerando projeções corrigidas (Modelagem de Tração)...")
history_matrix = reconstruir_caminho_inteligente(receita_inicial, sim_final_mrr, meses, volatility=0.15)

# --- 3. CÁLCULO DAS FAIXAS (P10, P50, P90) ---
p10_curve = np.percentile(history_matrix, 10, axis=0)
p50_curve = np.percentile(history_matrix, 50, axis=0)
p90_curve = np.percentile(history_matrix, 90, axis=0)
mean_curve = np.mean(history_matrix, axis=0)
eixo_x = list(range(meses + 1))

# --- 4. GRÁFICO VISUAL (CONE) ---
fig = go.Figure()

# Faixa de Incerteza
fig.add_trace(go.Scatter(
    x=eixo_x + eixo_x[::-1],
    y=list(p90_curve) + list(p10_curve)[::-1],
    fill='toself', fillcolor='rgba(180, 180, 180, 0.2)', line=dict(width=0),
    name='Área de Risco (Cenários Prováveis)'
))

# Linhas
fig.add_trace(go.Scatter(x=eixo_x, y=p10_curve, name='Pessimista (P10)', line=dict(color=cor_pessimista, dash='dot')))
fig.add_trace(go.Scatter(x=eixo_x, y=p50_curve, name='REALISTA (Meta)', line=dict(color=cor_base, width=4)))
fig.add_trace(go.Scatter(x=eixo_x, y=p90_curve, name='Otimista (P90)', line=dict(color=cor_otimista, dash='dot')))

fig.update_layout(
    title=f"<b>PROJEÇÃO DE TRAÇÃO MENSAL (MRR)</b>",
    template="plotly_white", height=500, hovermode="x unified",
    yaxis=dict(tickformat=",.0f", title="Receita Mensal (R$)"),
    xaxis=dict(title="Mês de Operação")
)
fig.show()

# --- 5. TABELA EXPLICATIVA PARA O PITCH ---
df_pitch = pd.DataFrame({
    'Mês': range(meses + 1),
    'Pessimista': p10_curve,
    'Realista (Base)': p50_curve,
    'Otimista': p90_curve,
    'Risco ($)': p90_curve - p10_curve
})

# Função para formatar moeda limpa
def fmt(x): return f"{x:,.2f}"

# Criar DataFrame formatado (apenas strings para exibição)
df_display = df_pitch.copy()
for col in ['Pessimista', 'Realista (Base)', 'Otimista', 'Risco ($)']:
    df_display[col] = df_display[col].apply(fmt)

# Exibição do Glossário e Tabela
print("\n" + "="*90)
print("📘 GLOSSÁRIO PARA O PITCH (O que dizer ao Investidor):")
print("="*90)
print("1. PESSIMISTA (P10): 'Mesmo num cenário ruim, onde a tração é 80% menor que o esperado, chegamos neste valor.'")
print("2. REALISTA (P50): 'Esta é a nossa meta oficial. É o centro estatístico da simulação.'")
print("3. OTIMISTA (P90): 'Se o produto viralizar ou o CAC cair pela metade, este é o potencial de escala.'")
print("4. RISCO ($): 'A diferença entre o otimista e o pessimista. Mostra o tamanho da incerteza no longo prazo.'")
print("-" * 90)
print(f"📊 TABELA DE DADOS MENSAL ({meses} Meses) - Copie para o Excel")
print("-" * 90)

# Mostrar tabela inteligente (Primeiros meses + Últimos meses)
# Mostra os meses 1, 2, 3, 6, 12, 18, 24... para não poluir, ou tudo se for curto
indices_mostrar = [0, 1, 2, 3, 4, 5, 6] + [m for m in [12, 18, 24, 30, 36, 48, 60] if m <= meses]
indices_mostrar = sorted(list(set(indices_mostrar))) # Remove duplicatas e ordena
indices_mostrar = [i for i in indices_mostrar if i < len(df_display)] # Garante que está no range

print(df_display.iloc[indices_mostrar].to_markdown(index=False))

print("\n💡 NOTA: Os valores 'centavos' foram eliminados. A curva agora simula vendas reais desde o mês 1.")

🔄 Gerando projeções corrigidas (Modelagem de Tração)...



📘 GLOSSÁRIO PARA O PITCH (O que dizer ao Investidor):
1. PESSIMISTA (P10): 'Mesmo num cenário ruim, onde a tração é 80% menor que o esperado, chegamos neste valor.'
2. REALISTA (P50): 'Esta é a nossa meta oficial. É o centro estatístico da simulação.'
3. OTIMISTA (P90): 'Se o produto viralizar ou o CAC cair pela metade, este é o potencial de escala.'
4. RISCO ($): 'A diferença entre o otimista e o pessimista. Mostra o tamanho da incerteza no longo prazo.'
------------------------------------------------------------------------------------------
📊 TABELA DE DADOS MENSAL (36 Meses) - Copie para o Excel
------------------------------------------------------------------------------------------
|   Mês | Pessimista   | Realista (Base)   | Otimista   | Risco ($)   |
|------:|:-------------|:------------------|:-----------|:------------|
|     0 | 0.00         | 0.00              | 0.00       | 0.00        |
|     1 | 0.00         | 521.12            | 4,663.06   | 4,663.06    |
|     2 | 0.

In [15]:
# ============================================================================
# CELULA 15
# ============================================================================
# BREAK-EVEN - Gráfico de Barras por Mês (MELHORADO & DINÂMICO)
# ============================================================================

breakeven_data = mc_results['Break_Even_Mes'].copy()
# Filtrar apenas os que atingiram break-even dentro do período projetado
breakeven_data = breakeven_data[breakeven_data <= PREMISSAS['meses_projecao']]

p10_be = np.percentile(breakeven_data, 10) if len(breakeven_data) > 0 else 0
p50_be = np.percentile(breakeven_data, 50) if len(breakeven_data) > 0 else 0
p90_be = np.percentile(breakeven_data, 90) if len(breakeven_data) > 0 else 0

# Definição dinâmica do eixo X (Visualizar até mês 24 ou até o max encontrado)
max_view = min(PREMISSAS['meses_projecao'], max(24, int(p90_be) + 2))
meses_unicos = np.arange(1, max_view + 1)

contagem_por_mes = []
for mes in meses_unicos:
    count = (breakeven_data == mes).sum()
    contagem_por_mes.append(count)

# Calcular porcentagens
total = len(breakeven_data)
pct_por_mes = [(c / total * 100) if total > 0 else 0 for c in contagem_por_mes]

# Cores baseadas nos percentis (Dinâmico)
cores = []
for mes in meses_unicos:
    if mes <= p10_be:
        cores.append(PREMISSAS['cores']['success'])  # Rápido
    elif mes <= p50_be:
        cores.append(PREMISSAS['cores']['primary'])  # Típico
    elif mes <= p90_be:
        cores.append(PREMISSAS['cores']['warning'])  # Lento
    else:
        cores.append(PREMISSAS['cores']['danger'])   # Muito lento

fig_be = go.Figure()

fig_be.add_trace(go.Bar(
    x=meses_unicos,
    y=contagem_por_mes,
    text=[f'{p:.1f}%' if p > 1 else '' for p in pct_por_mes], # Só mostra texto se > 1%
    textposition='outside',
    marker=dict(
        color=cores,
        line=dict(color='white', width=1.5)
    ),
    hovertemplate='<b>Mês %{x}</b><br>' +
                  'Cenários: %{y}<br>' +
                  'Porcentagem: %{text}<extra></extra>'
))

fig_be.update_layout(
    title={
        'text': '<b>EM QUAL MÊS O NEGÓCIO COMEÇA A DAR LUCRO?</b><br>' +
                f"<sup>Break-Even: quando receita supera custos totais ({len(breakeven_data)} cenários lucrativos)</sup>",
        'x': 0.5,
        'xanchor': 'center',
        'y': 0.95,
        'yanchor': 'top',
    },
    xaxis=dict(
        title='<b>Mês do Break-Even</b>',
        dtick=1,
        range=[0.5, max_view + 0.5]
    ),
    yaxis=dict(title='<b>Quantidade de Cenários</b>'),
    template='plotly_white',
    height=PREMISSAS['layout_altura_grafico'],
    font=dict(size=12),
    margin=dict(t=120, b=80, l=80, r=100),
    showlegend=False
)

# Anotações com percentis (Posicionamento dinâmico)
if p10_be > 0:
    fig_be.add_annotation(
        text=f"<b>10% Mais Rápidos</b><br>Até Mês {int(p10_be)}",
        xref="paper", yref="paper", x=0.15, y=1.08,
        showarrow=False, font=dict(size=11),
        bgcolor='white', bordercolor=PREMISSAS['cores']['success'], borderwidth=2, borderpad=6
    )

if p50_be > 0:
    fig_be.add_annotation(
        text=f"<b>50% Típico</b><br>Até Mês {int(p50_be)}",
        xref="paper", yref="paper", x=0.5, y=1.08,
        showarrow=False, font=dict(size=12),
        bgcolor='white', bordercolor=PREMISSAS['cores']['primary'], borderwidth=3, borderpad=6
    )

if p90_be > 0:
    fig_be.add_annotation(
        text=f"<b>90% dos Cenários</b><br>Até Mês {int(p90_be)}",
        xref="paper", yref="paper", x=0.85, y=1.08,
        showarrow=False, font=dict(size=11),
        bgcolor='white', bordercolor=PREMISSAS['cores']['warning'], borderwidth=2, borderpad=6
    )

fig_be.show()

print(f"\nBreak-Even (Cenários que atingiram lucro):")
print(f"  • 10% mais rápidos: lucro até mês {int(p10_be)}")
print(f"  • 50% dos cenários (típico): lucro até mês {int(p50_be)}")
print(f"  • 90% dos cenários: lucro até mês {int(p90_be)}")
print(f"  • Cenários sem lucro em {PREMISSAS['meses_projecao']} meses: {PREMISSAS['mc_n_simulacoes'] - len(breakeven_data)}")

print("✅ CELULA 14: Hardcoded removidos (eixos dinâmicos, cores)")


Break-Even (Cenários que atingiram lucro):
  • 10% mais rápidos: lucro até mês 5
  • 50% dos cenários (típico): lucro até mês 6
  • 90% dos cenários: lucro até mês 10
  • Cenários sem lucro em 36 meses: 235
✅ CELULA 14: Hardcoded removidos (eixos dinâmicos, cores)


In [16]:
# ============================================================================
# CELULA 16 - DASHBOARD EXECUTIVO - KPIs + RECOMENDAÇÕES ESTRATÉGICAS
# ============================================================================
print("="*80)
print("🎯 DASHBOARD EXECUTIVO - ANÁLISE DE SAÚDE DO NEGÓCIO")
print("="*80 + "\n")

# === COLETAR KPIS CHAVE ===
kpis = {
    'churn': PREMISSAS['churn_base'] * 100,
    'ltv_cac_ratio': df_base['LTV_CAC_Ratio'].iloc[-1],
    'ltv': df_base['LTV'].iloc[-1],
    'cac': df_base['CAC'].iloc[-1],
    'payback_meses': df_base['Payback_Meses'].iloc[-1],
    'margem_bruta': df_base['Margem_Bruta_Pct'].iloc[-1],
    'conv_trial_pagante': (df_base['Novos_Pagantes'].sum() / df_base['Trials'].sum() * 100) if df_base['Trials'].sum() > 0 else 0,
    'mrr_final': df_base['MRR'].iloc[-1],
    'crescimento_mrr': ((df_base['MRR'].iloc[-1] / df_base['MRR'].iloc[0]) - 1) * 100 if df_base['MRR'].iloc[0] > 0 else 0,
    'usuarios_final': df_base['Usuarios_Finais'].iloc[-1],
    'caixa_final': df_base['Saldo_Caixa'].iloc[-1]
}

# === MATRIZ DE SAÚDE (Usando Benchmarks da PREMISSAS) ===
saude = {
    '🔴 CRÍTICO': [],
    '🟡 ATENÇÃO': [],
    '🟢 SAUDÁVEL': []
}

# Análise Churn
if kpis['churn'] > PREMISSAS['benchmark_churn_critico_pct']:
    saude['🔴 CRÍTICO'].append(f"Churn: {kpis['churn']:.1f}%/mês (Meta: <{PREMISSAS['benchmark_churn_atencao_pct']}%)")
elif kpis['churn'] > PREMISSAS['benchmark_churn_atencao_pct']:
    saude['🟡 ATENÇÃO'].append(f"Churn: {kpis['churn']:.1f}%/mês (Meta: <{PREMISSAS['benchmark_churn_atencao_pct']}%)")
else:
    saude['🟢 SAUDÁVEL'].append(f"Churn: {kpis['churn']:.1f}%/mês")

# Análise LTV/CAC
if kpis['ltv_cac_ratio'] < PREMISSAS['benchmark_ltv_cac_atencao']:
    saude['🔴 CRÍTICO'].append(f"LTV/CAC: {kpis['ltv_cac_ratio']:.1f}x (Meta: {PREMISSAS['benchmark_ltv_cac_atencao']}-{PREMISSAS['benchmark_ltv_cac_excelente']}x)")
elif kpis['ltv_cac_ratio'] < PREMISSAS['benchmark_ltv_cac_excelente']:
    saude['🟡 ATENÇÃO'].append(f"LTV/CAC: {kpis['ltv_cac_ratio']:.1f}x (Pode melhorar)")
else:
    saude['🟢 SAUDÁVEL'].append(f"LTV/CAC: {kpis['ltv_cac_ratio']:.1f}x (Excelente!)")

# Análise Payback
if kpis['payback_meses'] > PREMISSAS['benchmark_payback_max_meses']:
    saude['🟡 ATENÇÃO'].append(f"Payback: {kpis['payback_meses']:.1f} meses (Meta: <{PREMISSAS['benchmark_payback_max_meses']:.0f})")
elif kpis['payback_meses'] < 3:
    saude['🟡 ATENÇÃO'].append(f"Payback: {kpis['payback_meses']:.1f} meses (Suspeito - verificar)")
else:
    saude['🟢 SAUDÁVEL'].append(f"Payback: {kpis['payback_meses']:.1f} meses")

# Análise Margem Bruta
if kpis['margem_bruta'] < PREMISSAS['benchmark_margem_critica_pct']:
    saude['🔴 CRÍTICO'].append(f"Margem Bruta: {kpis['margem_bruta']:.1f}% (Meta: >{PREMISSAS['benchmark_margem_critica_pct']}%)")
elif kpis['margem_bruta'] < PREMISSAS['benchmark_margem_alvo_pct']:
    saude['🟡 ATENÇÃO'].append(f"Margem Bruta: {kpis['margem_bruta']:.1f}% (Pode otimizar)")
else:
    saude['🟢 SAUDÁVEL'].append(f"Margem Bruta: {kpis['margem_bruta']:.1f}%")

# Análise Conversão
if kpis['conv_trial_pagante'] < PREMISSAS['benchmark_conversao_trial_min_pct']:
    saude['🔴 CRÍTICO'].append(f"Trial→Pagante: {kpis['conv_trial_pagante']:.1f}% (Meta: >{PREMISSAS['benchmark_conversao_trial_min_pct']}%)")
elif kpis['conv_trial_pagante'] < PREMISSAS['benchmark_conversao_trial_alvo_pct']:
    saude['🟡 ATENÇÃO'].append(f"Trial→Pagante: {kpis['conv_trial_pagante']:.1f}% (Pode melhorar)")
else:
    saude['🟢 SAUDÁVEL'].append(f"Trial→Pagante: {kpis['conv_trial_pagante']:.1f}%")

# Análise Caixa
if kpis['caixa_final'] < 0:
    saude['🔴 CRÍTICO'].append(f"Caixa Final: {formatar_moeda(kpis['caixa_final'])} (NEGATIVO!)")
elif kpis['caixa_final'] < PREMISSAS['meta_caixa_seguranca']:
    saude['🟡 ATENÇÃO'].append(f"Caixa Final: {formatar_moeda(kpis['caixa_final'])} (Baixo)")
else:
    saude['🟢 SAUDÁVEL'].append(f"Caixa Final: {formatar_moeda(kpis['caixa_final'])}")

# Exibir matriz
for nivel, itens in saude.items():
    if itens:
        print(f"\n{nivel}:")
        for item in itens:
            print(f"  • {item}")
print("\n" + "="*80)

# === RECOMENDAÇÕES ESTRATÉGICAS (CONDICIONAIS) ===
print("\n💡 RECOMENDAÇÕES ESTRATÉGICAS\n")
recomendacoes = []

# 1. Churn
if kpis['churn'] > PREMISSAS['benchmark_churn_atencao_pct']:
    recomendacoes.append({
        'prioridade': 1,
        'kpi': 'Churn',
        'problema': f"Churn de {kpis['churn']:.1f}%/mês está alto",
        'acoes': [
            "Implementar onboarding estruturado (primeiros 7 dias críticos)",
            "Criar sistema de health score para identificar usuários em risco",
            "Analisar motivos de cancelamento (pesquisa de exit)",
            f"💰 IMPACTO: Reduzir para 4% = +{(PREMISSAS['meta_mrr']/100):.0f} usuários retidos"
        ]
    })

# 2. LTV/CAC
if kpis['ltv_cac_ratio'] < PREMISSAS['benchmark_ltv_cac_atencao']:
    recomendacoes.append({
        'prioridade': 1,
        'kpi': 'LTV/CAC',
        'problema': f"Relação LTV/CAC de {kpis['ltv_cac_ratio']:.1f}x está abaixo do mínimo ({PREMISSAS['benchmark_ltv_cac_atencao']}x)",
        'acoes': [
            "OPÇÃO A: Reduzir CAC (otimizar canais, SEO, referral)",
            "OPÇÃO B: Aumentar LTV (upsell, cross-sell, reduzir churn)",
            "Testar diferentes canais de aquisição (orgânico vs pago)",
            f"💰 IMPACTO: Atingir {PREMISSAS['benchmark_ltv_cac_atencao']}x = Negócio sustentável"
        ]
    })

# 3. Conversão
if kpis['conv_trial_pagante'] < PREMISSAS['benchmark_conversao_trial_alvo_pct']:
    recomendacoes.append({
        'prioridade': 2,
        'kpi': 'Conversão Trial→Pagante',
        'problema': f"Taxa de {kpis['conv_trial_pagante']:.1f}% tem espaço para otimização",
        'acoes': [
            "Melhorar experiência do trial (remover fricções)",
            "Adicionar prova social (depoimentos, cases)",
            "Oferecer suporte proativo durante trial",
            f"💰 IMPACTO: Aumentar para {PREMISSAS['benchmark_conversao_trial_alvo_pct']}% = +20-30% de clientes"
        ]
    })

# 4. Crescimento (Comparando com a Meta definida)
if kpis['crescimento_mrr'] < PREMISSAS['meta_crescimento_mrr_pct']:
    recomendacoes.append({
        'prioridade': 3,
        'kpi': 'Crescimento MRR',
        'problema': f"Crescimento de {kpis['crescimento_mrr']:.0f}% em 36m está abaixo da meta ({PREMISSAS['meta_crescimento_mrr_pct']}%)",
        'acoes': [
            "Acelerar aquisição (aumentar budget de marketing se LTV/CAC saudável)",
            "Implementar programa de referral (viralidade)",
            "Expandir para novos segmentos/personas"
        ]
    })

# 5. Margem
if kpis['margem_bruta'] < PREMISSAS['benchmark_margem_alvo_pct']:
    recomendacoes.append({
        'prioridade': 2,
        'kpi': 'Margem Bruta',
        'problema': f"Margem de {kpis['margem_bruta']:.1f}% limita escalabilidade",
        'acoes': [
            "Renegociar custos de IA (desconto por volume com OpenAI)",
            "Otimizar uso de tokens (cache, compressão de contexto)",
            f"💰 IMPACTO: +5pp de margem = +{formatar_moeda(kpis['mrr_final'] * 0.05)} de lucro/mês"
        ]
    })

# Ordenar por prioridade e exibir
recomendacoes.sort(key=lambda x: x['prioridade'])
for i, rec in enumerate(recomendacoes, 1):
    print(f"🎯 RECOMENDAÇÃO {i} - {rec['kpi'].upper()}")
    print(f"   Prioridade: {'🔥 URGENTE' if rec['prioridade'] == 1 else '⚠️ IMPORTANTE' if rec['prioridade'] == 2 else '💡 OTIMIZAÇÃO'}")
    print(f"   Problema: {rec['problema']}")
    print("   Ações:")
    for acao in rec['acoes']:
        print(f"      → {acao}")
    print()

print("="*80)
print("✅ CELULA 15: Hardcoded removidos (benchmarks e metas)")

🎯 DASHBOARD EXECUTIVO - ANÁLISE DE SAÚDE DO NEGÓCIO


🔴 CRÍTICO:
  • Churn: 8.0%/mês (Meta: <5.0%)

🟡 ATENÇÃO:
  • Payback: 2.4 meses (Suspeito - verificar)
  • Trial→Pagante: 11.8% (Pode melhorar)

🟢 SAUDÁVEL:
  • LTV/CAC: 5.1x (Excelente!)
  • Margem Bruta: 84.3%
  • Caixa Final: R$ 116.401,14


💡 RECOMENDAÇÕES ESTRATÉGICAS

🎯 RECOMENDAÇÃO 1 - CHURN
   Prioridade: 🔥 URGENTE
   Problema: Churn de 8.0%/mês está alto
   Ações:
      → Implementar onboarding estruturado (primeiros 7 dias críticos)
      → Criar sistema de health score para identificar usuários em risco
      → Analisar motivos de cancelamento (pesquisa de exit)
      → 💰 IMPACTO: Reduzir para 4% = +500 usuários retidos

🎯 RECOMENDAÇÃO 2 - CONVERSÃO TRIAL→PAGANTE
   Prioridade: ⚠️ IMPORTANTE
   Problema: Taxa de 11.8% tem espaço para otimização
   Ações:
      → Melhorar experiência do trial (remover fricções)
      → Adicionar prova social (depoimentos, cases)
      → Oferecer suporte proativo durante trial
      → 💰 IMP

In [19]:
# ============================================================================
# CELULA 17: TABELA EXECUTIVA (CORRIGIDA E ALINHADA COM CELULA 05)
# ============================================================================
print("="*100)
print("📊 TABELA EXECUTIVA COMPLETA - PROJEÇÃO FINANCEIRA SAM (36 MESES)")
print("="*100 + "\n")

# Determinar se usamos métricas ajustadas
usar_ajustado = PREMISSAS.get('modelo_afiliado_habilitado', False) and 'Lucro_Liquido_Ajustado' in df_base.columns

# Índices
idx_m1 = 0
idx_m12 = PREMISSAS['meses_por_ano'] - 1
idx_m36 = -1

# --- CORREÇÃO DE SEGURANÇA (Igual à Célula 05) ---
# Calcula o investimento somando exatamente o que saiu do caixa na simulação
investimento_real_simulado = PREMISSAS['capex'] + df_base['Aportes'].sum()
# ------------------------------------------------

# Funil
trafego_m1 = df_base['Trafego'].iloc[idx_m1]
trafego_m12 = df_base['Trafego'].iloc[idx_m12]
trafego_m36 = df_base['Trafego'].iloc[idx_m36]
trials_total = df_base['Trials'].sum()
pagantes_total = df_base['Novos_Pagantes'].sum()
conv_trial_pagante = (pagantes_total / trials_total * 100) if trials_total > 0 else 0

# Usuários
usuarios_m12 = df_base['Usuarios_Finais'].iloc[idx_m12]
usuarios_m36 = df_base['Usuarios_Finais'].iloc[idx_m36]
churn_medio = PREMISSAS['churn_base'] * 100

# Receita
mrr_m1 = df_base['MRR'].iloc[idx_m1]
mrr_m12 = df_base['MRR'].iloc[idx_m12]
mrr_m36 = df_base['MRR'].iloc[idx_m36]
arr_m36 = mrr_m36 * PREMISSAS['meses_por_ano']

# Unit Economics
cac_m12 = df_base['CAC'].iloc[idx_m12]
cac_m36 = df_base['CAC'].iloc[idx_m36]
ltv_m36 = df_base['LTV'].iloc[idx_m36]
ltv_cac_m36 = df_base['LTV_CAC_Ratio'].iloc[idx_m36]
payback_m36 = df_base['Payback_Meses'].iloc[idx_m36]

# Margens e Custos
margem_bruta_m36 = df_base['Margem_Bruta_Pct'].iloc[idx_m36]

# Seleção de variáveis (Ajustado vs Normal)
if usar_ajustado:
    custo_total_m36 = df_base['COGS_Total'].iloc[idx_m36] + df_base['OPEX_Total_Ajustado'].iloc[idx_m36]
    lucro_liquido_m36 = df_base['Lucro_Liquido_Ajustado'].iloc[idx_m36]
    retorno_investidor = df_base['Dist_Investidor_Ajustado'].sum()
    caixa_m36 = df_base['Saldo_Caixa_Ajustado'].iloc[idx_m36]
else:
    custo_total_m36 = df_base['COGS_Total'].iloc[idx_m36] + df_base['OPEX_Total'].iloc[idx_m36]
    lucro_liquido_m36 = df_base['Lucro_Liquido'].iloc[idx_m36]
    retorno_investidor = df_base['Dist_Investidor'].sum()
    caixa_m36 = df_base['Saldo_Caixa'].iloc[idx_m36]

custos_pct_receita = (custo_total_m36 / mrr_m36 * 100) if mrr_m36 > 0 else 0

# Financeiro
mes_breakeven = df_base[df_base['Lucro_Liquido'] > 0]['Mes'].min() if any(df_base['Lucro_Liquido'] > 0) else None
caixa_min = df_base['Saldo_Caixa'].min()

# ROI Calculado sobre o Investimento Real da Simulação
roi_investidor = ((retorno_investidor / investimento_real_simulado) - 1) * 100 if investimento_real_simulado > 0 else 0

# Valuation
val_conservador = arr_m36 * 5
val_otimista = arr_m36 * 10

# Construção da Tabela
tabela_dados = []

def criar_linha(categoria, metrica, m1, m12, m36, benchmark, status):
    return {
        'Categoria': categoria,
        'Métrica': metrica,
        'Mês 1': m1,
        'Mês 12': m12,
        'Mês 36': m36,
        'Benchmark': benchmark,
        'Status': status
    }

tabela_dados.append(criar_linha('🌐 FUNIL', 'Tráfego Mensal', f"{trafego_m1:,.0f}", f"{trafego_m12:,.0f}", f"{trafego_m36:,.0f}", "Crescimento", '✅'))
tabela_dados.append(criar_linha('🌐 FUNIL', 'Conversão Trial→Pagante', '-', '-', f"{conv_trial_pagante:.1f}%", f"Meta: >{PREMISSAS['benchmark_conversao_trial_min_pct']}%", '⚠️' if conv_trial_pagante < PREMISSAS['benchmark_conversao_trial_alvo_pct'] else '✅'))
tabela_dados.append(criar_linha('👥 USUÁRIOS', 'Usuários Pagantes', '-', f"{usuarios_m12:.0f}", f"{usuarios_m36:.0f}", 'Crescimento', '✅'))
tabela_dados.append(criar_linha('👥 USUÁRIOS', 'Churn Mensal', f"{churn_medio:.1f}%", f"{churn_medio:.1f}%", f"{churn_medio:.1f}%", f"Meta: <{PREMISSAS['benchmark_churn_atencao_pct']}%", '⚠️' if churn_medio > PREMISSAS['benchmark_churn_atencao_pct'] else '✅'))
tabela_dados.append(criar_linha('💰 RECEITA', 'MRR', formatar_moeda(mrr_m1), formatar_moeda(mrr_m12), formatar_moeda(mrr_m36), f"Meta: >{formatar_moeda(PREMISSAS['meta_mrr'])}", '✅'))
tabela_dados.append(criar_linha('💰 RECEITA', 'ARR (Anualizado)', '-', '-', formatar_moeda(arr_m36), f"Meta: >{formatar_moeda(PREMISSAS['meta_arr'])}", '✅'))
tabela_dados.append(criar_linha('📈 UNIT ECON', 'CAC', '-', formatar_moeda(cac_m12), formatar_moeda(cac_m36), f"Ideal < {formatar_moeda(PREMISSAS['benchmark_cac_ideal_trading'])}", '✅'))
tabela_dados.append(criar_linha('📈 UNIT ECON', 'LTV', '-', '-', formatar_moeda(ltv_m36), f"Min: {formatar_moeda(PREMISSAS['benchmark_ltv_fintech_min'])}", '✅'))
tabela_dados.append(criar_linha('📈 UNIT ECON', 'LTV/CAC', '-', '-', f"{ltv_cac_m36:.1f}x", f"{PREMISSAS['benchmark_ltv_cac_atencao']}x-{PREMISSAS['benchmark_ltv_cac_excelente']}x", '🟢' if ltv_cac_m36 >= 5 else '✅'))
tabela_dados.append(criar_linha('📈 UNIT ECON', 'Payback', '-', '-', f"{payback_m36:.1f} meses", f"<{PREMISSAS['benchmark_payback_max_meses']:.0f} meses", '✅'))
tabela_dados.append(criar_linha('💸 MARGENS', 'Margem Bruta', '-', '-', f"{margem_bruta_m36:.1f}%", f"Meta: >{PREMISSAS['benchmark_margem_alvo_pct']}%", '✅'))
tabela_dados.append(criar_linha('💵 FINANCEIRO', 'Break-Even', '-', '-', f"Mês {int(mes_breakeven)}" if mes_breakeven else "-", "Meta: <12m", '✅'))
tabela_dados.append(criar_linha('💵 FINANCEIRO', 'Saldo Caixa Final', '-', '-', formatar_moeda(caixa_m36), f"Meta: >{formatar_moeda(PREMISSAS['meta_caixa_ideal'])}", '✅' if caixa_m36 > PREMISSAS['meta_caixa_seguranca'] else '⚠️'))

# Linha Investidor - Agora usando a mesma base da Célula 05
tabela_dados.append(criar_linha('🤑 INVESTIDOR', 'ROI (Retorno)', '-', '-', f"{roi_investidor:.1f}%", f"Base Inv: {formatar_moeda(investimento_real_simulado)}", '✅'))

tabela_dados.append(criar_linha('💎 VALUATION', 'Valor Estimado', '-', '-', f"{formatar_moeda(val_conservador)}", "5x ARR", 'ℹ️'))

df_tabela_final = pd.DataFrame(tabela_dados)
print(df_tabela_final.to_string(index=False))
print("\n" + "="*100)

📊 TABELA EXECUTIVA COMPLETA - PROJEÇÃO FINANCEIRA SAM (36 MESES)

   Categoria                 Métrica       Mês 1       Mês 12          Mês 36              Benchmark Status
     🌐 FUNIL          Tráfego Mensal       2,687        4,595          20,314            Crescimento      ✅
     🌐 FUNIL Conversão Trial→Pagante           -            -           11.8%           Meta: >10.0%     ⚠️
  👥 USUÁRIOS       Usuários Pagantes           -          155            1269            Crescimento      ✅
  👥 USUÁRIOS            Churn Mensal        8.0%         8.0%            8.0%            Meta: <5.0%     ⚠️
   💰 RECEITA                     MRR R$ 2.435,86 R$ 13.775,30   R$ 112.821,88    Meta: >R$ 50.000,00      ✅
   💰 RECEITA        ARR (Anualizado)           -            - R$ 1.353.862,56   Meta: >R$ 600.000,00      ✅
 📈 UNIT ECON                     CAC           -    R$ 222,80       R$ 182,35      Ideal < R$ 200,00      ✅
 📈 UNIT ECON                     LTV           -            -       R$

In [18]:
# ============================================================================
# CELULA 18 - CAPTURA E EXPORTAÇÃO DE RESULTADOS
# ============================================================================
import json
import os
import numpy as np

# Dicionário para acumular resultados
auditoria_resultados = {
    "simulacao_base": {},
    "monte_carlo": {},
    "resumo_executivo": {}
}

# 1. Capturar Resultados da Simulação Base (df_base)
try:
    if 'df_base' in locals():
        # Converter últimas linhas para dict
        last_row = df_base.tail(1).to_dict(orient='records')[0]
        # Converter Timestamps para string
        for k, v in last_row.items():
            if isinstance(v, pd.Timestamp):
                last_row[k] = str(v)
                
        auditoria_resultados["simulacao_base"] = last_row
        # Adicionar totais
        auditoria_resultados["simulacao_base"]["Total_Receita"] = float(df_base['Receita_Mensal'].sum())
        auditoria_resultados["simulacao_base"]["Total_Lucro"] = float(df_base['Lucro_Liquido'].sum())
        print("✅ Dados da Simulação Base capturados.")
    else:
        print("⚠️ df_base não encontrado.")
except Exception as e:
        print(f"❌ Erro ao capturar df_base: {e}")

# 2. Capturar Resultados Monte Carlo (mc_results)
try:
    if 'mc_results' in locals():
        # Resumir estatísticas
        mc_summary = {}
        for k, v in mc_results.items():
            # Tratamento seguro de arrays numpy
            if hasattr(v, 'tolist'):
                v = v.tolist()
            if isinstance(v, list) or isinstance(v, np.ndarray):
                clean_v = np.array(v)
                # Remover NaNs e Infinitos
                clean_v = clean_v[np.isfinite(clean_v)]
                
                if len(clean_v) > 0:
                    mc_summary[k] = {
                        "min": float(np.min(clean_v)),
                        "max": float(np.max(clean_v)),
                        "mean": float(np.mean(clean_v)),
                        "p10": float(np.percentile(clean_v, 10)),
                        "p50": float(np.percentile(clean_v, 50)),
                        "p90": float(np.percentile(clean_v, 90))
                    }
        auditoria_resultados["monte_carlo"] = mc_summary
        print("✅ Dados de Monte Carlo capturados.")
    else:
        print("⚠️ mc_results não encontrado.")
except Exception as e:
        print(f"❌ Erro ao capturar Monte Carlo: {e}")

# 3. Capturar Resumo Executivo (df_tabela_final)
try:
    if 'df_tabela_final' in locals():
        auditoria_resultados["resumo_executivo"] = df_tabela_final.to_dict(orient='records')
        print("✅ Dados do Resumo Executivo capturados.")
    else:
        print("⚠️ df_tabela_final não encontrado.")
except Exception as e:
        print(f"❌ Erro ao capturar Resumo Executivo: {e}")

# 4. Salvar em Arquivo
arquivo_saida = "relatorio_auditoria_resultados.json"
try:
    with open(arquivo_saida, "w", encoding="utf-8") as f:
        json.dump(auditoria_resultados, f, indent=4, default=str)
    print(f"\n💾 Relatório salvo com sucesso em: {os.path.abspath(arquivo_saida)}")
    
    # Exibir prévia
    print("\n--- PRÉVIA DOS DADOS CAPTURADOS ---")
    print(json.dumps(auditoria_resultados, indent=2, default=str)[:500] + "...")
except Exception as e:
    print(f"❌ Erro ao salvar arquivo: {e}")

✅ Dados da Simulação Base capturados.
✅ Dados de Monte Carlo capturados.
✅ Dados do Resumo Executivo capturados.

💾 Relatório salvo com sucesso em: e:\Projetos\sam_analise_financeira\notebooks\relatorio_auditoria_resultados.json

--- PRÉVIA DOS DADOS CAPTURADOS ---
{
  "simulacao_base": {
    "Data": "2028-10-01 00:00:00",
    "Mes": 36,
    "Trafego": 20314,
    "Trials": 1112,
    "Novos_Pagantes": 133,
    "Novos_Via_Pago": 109,
    "Novos_Via_Organico": 24,
    "Pct_Organico": 18.045112781954884,
    "Deficit_Meta": 0,
    "Usuarios_Finais": 1269.0875141899185,
    "Usuarios_Iniciais": 1234.8777328151289,
    "Usuarios_Perdidos": 98.79021862521031,
    "Usuarios_Lite": 761.4525085139511,
    "Usuarios_Trader": 380.72625425697555,
    "Usuarios_Pro": 12...
